# Eight-regime PDE benchmark and diffusion–sorption data contract

This notebook is a cleaned record of the original training program. Execution outputs and notebook execution metadata were removed; computational cells were preserved to maintain the historical experiment protocol. Dataset access and optional cache paths may require configuration.


In [ ]:

# ============================================================
# 0. Runtime
# ============================================================
import os, sys, math, time, json, hashlib, shutil, subprocess, zipfile, copy, gc, re
from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import h5py
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda", "A2-R1 FIX1 requires CUDA."

ROOT=Path(os.getenv("CIDM_BENCH_ROOT",str(Path.cwd())))
OUT=ROOT/"CIDM_Foundation_A2_R1_DiffSorp1D_FIX_RESULTS"
SRC=OUT/"sources"
FROZEN=OUT/"frozen"
CKPT=OUT/"checkpoints"
FIG=OUT/"figures"
FIGDATA=OUT/"figure_data"
PRED=OUT/"prediction_examples"
CACHE_ROOT=Path(os.getenv(
    "CIDM_A2_R1_CACHE_ROOT",
    "/content/CIDM_A2_R1_DiffSorp1D_Cache"
))
BASE_A2_RESULTS=OUT/"base_A2_results"

for p in [OUT,SRC,FROZEN,CKPT,FIG,FIGDATA,PRED,CACHE_ROOT,BASE_A2_RESULTS]:
    p.mkdir(parents=True,exist_ok=True)

HISTORY=4
HORIZON=6
TARGET_PARAMS=500_000
PARAM_TOL=0.05
SEEDS=[17,41,73]
DEV_SEED=17

MAX_UPDATES=int(os.getenv("CIDM_A2_R1_MAX_UPDATES","2500"))
MIN_UPDATES=int(os.getenv("CIDM_A2_R1_MIN_UPDATES","1000"))
EVAL_EVERY=int(os.getenv("CIDM_A2_R1_EVAL_EVERY","100"))
PATIENCE_EVALS=int(os.getenv("CIDM_A2_R1_PATIENCE_EVALS","6"))
REL_MIN_DELTA=float(os.getenv("CIDM_A2_R1_REL_MIN_DELTA","0.001"))

BATCH_1D=int(os.getenv("CIDM_A2_R1_BATCH_1D","16"))
WEIGHT_DECAY=1e-4

MODELS=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "NeuralOperator-FNO",
    "Transolver",
]

TASKS={
    "diff_sorp_1d":{
        "family":"diffusion_sorption",
        "filename":"1D_diff-sorp_NA_NA.h5",
        "url":"https://darus.uni-stuttgart.de/api/access/datafile/133020",
        "md5":"9d466d1213065619d087319e16d9a938",
        "spatial_stride":4,
        "expected_dim":1,
        "expected_channels":1,
        "expected_spatial":[256],
    }
}

print("GPU:",torch.cuda.get_device_name(0))
print("Torch:",torch.__version__)
print("OUT:",OUT)

REPORT={
    "experiment":"CIDM Foundation A2-R1 Diffusion-Sorption 1D Contract Repair",
    "started_at":time.strftime("%Y-%m-%d %H:%M:%S"),
    "repair_reason":"A2 grouped parser interpreted official [T,X,1] as 2D [T,1,X,1]",
    "seeds":SEEDS,
}

def run(cmd,cwd=None,timeout=1800):
    p=subprocess.run(
        cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,timeout=timeout
    )
    return p.returncode,p.stdout

def md5_file(path,chunk=8*1024*1024):
    h=hashlib.md5()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def sha256_file(path,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def save_json(obj,path):
    Path(path).write_text(
        json.dumps(obj,ensure_ascii=False,indent=2,default=str),
        encoding="utf-8"
    )

def seed_all(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def portable_state_dict(model):
    raw=model.state_dict()
    out=OrderedDict()
    for k,v in raw.items():
        if k.split(".")[-1]=="_metadata":
            continue
        if torch.is_tensor(v):
            out[k]=v.detach().cpu().clone()
        else:
            out[k]=copy.deepcopy(v)
    if hasattr(raw,"_metadata"):
        out._metadata=copy.deepcopy(raw._metadata)
    return out

def load_portable_state_dict(model,sd):
    incompat=model.load_state_dict(sd,strict=False)
    missing=list(incompat.missing_keys)
    unexpected=[
        k for k in incompat.unexpected_keys
        if k.split(".")[-1]!="_metadata"
    ]
    if missing or unexpected:
        raise RuntimeError(
            f"checkpoint mismatch: missing={missing}, unexpected={unexpected}"
        )


In [ ]:

# ============================================================
# 1. Freeze audit:
#    CIDM Foundation Core v1.0 Final
#    + CIDM-Complete archived executable topology
# ============================================================
CIDM_CORE_SOURCE = '"""\nCIDM Foundation Core v1.0\n=========================\n\nFrozen canonical implementation for fair external benchmarking.\n\nThis file contains ONLY the universal core selected after the A1-R1-R3\ncomponent-attribution review:\n\n1. Finite-history predictive information state.\n2. Spatial endpoint encoder.\n3. Temporal-history encoder.\n4. Fusion into a single latent information state.\n5. Bounded local latent residual dynamics.\n6. Bounded positive step size.\n7. Pure latent rollout.\n8. Field decoder.\n\nNot part of the universal core:\n- phase/spectral encoder,\n- skew-symmetric channel rotation,\n- antisymmetric flux,\n- hidden reservoir,\n- causal forcing port,\n- transport branch,\n- cross-scale transfer,\n- BER/SOC,\n- state-sync regularization,\n- Cayley update.\n\nThose are task-specific or training/numerical optionals and must not be silently\nenabled in Foundation Core benchmark runs.\n\nImportant:\nThe numerical caps below are architectural hyperparameters, not physical\nconstants.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass(frozen=True)\nclass CIDMFoundationConfig:\n    spatial_dim: int = 2          # 1 or 2\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n\n    # Frozen benchmark defaults; architectural hyperparameters, not laws.\n    local_cap: float = 0.10\n    step_cap: float = 0.30\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef _conv_nd(dim: int):\n    if dim == 1:\n        return nn.Conv1d\n    if dim == 2:\n        return nn.Conv2d\n    raise ValueError(f"spatial_dim must be 1 or 2, got {dim}")\n\n\nclass ConvBlockND(nn.Module):\n    def __init__(self, dim: int, in_ch: int, out_ch: int, hidden: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            Conv(hidden, out_ch, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field X_t."""\n\n    def __init__(self, dim: int, channels: int, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, channels, latent_dim, hidden)\n\n    def forward(self, x_t: torch.Tensor) -> torch.Tensor:\n        return self.net(x_t)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode finite history H_t = X_{t-L+1:t}."""\n\n    def __init__(\n        self,\n        dim: int,\n        channels: int,\n        hist_len: int,\n        latent_dim: int,\n        hidden: int,\n    ):\n        super().__init__()\n        self.hist_len = hist_len\n        self.channels = channels\n        self.net = ConvBlockND(\n            dim,\n            channels * hist_len,\n            latent_dim,\n            hidden,\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: [B, L, C, *S]\n        if history.shape[1] != self.hist_len:\n            raise ValueError(\n                f"Expected history length {self.hist_len}, got {history.shape[1]}"\n            )\n        b, l, c, *spatial = history.shape\n        if c != self.channels:\n            raise ValueError(f"Expected {self.channels} channels, got {c}")\n        x = history.reshape(b, l * c, *spatial)\n        return self.net(x)\n\n\nclass InformationStateFusion(nn.Module):\n    """Fuse spatial endpoint and temporal-history views into Psi_t."""\n\n    def __init__(self, dim: int, latent_dim: int):\n        super().__init__()\n        Conv = _conv_nd(dim)\n        self.net = nn.Sequential(\n            Conv(2 * latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            Conv(latent_dim, latent_dim, 1),\n        )\n\n    def forward(\n        self,\n        z_space: torch.Tensor,\n        z_time: torch.Tensor,\n    ) -> torch.Tensor:\n        return self.net(torch.cat([z_space, z_time], dim=1))\n\n\nclass BoundedLocalResidualDynamics(nn.Module):\n    """\n    Minimal universal transition field.\n\n    The residual branch is bounded twice:\n    - branch responsibility: local_cap * sigmoid(local_logit)\n    - global step: step_cap * sigmoid(step_logit)\n\n    This prevents an unconstrained transition from silently dominating the\n    latent information state.\n    """\n\n    def __init__(\n        self,\n        dim: int,\n        latent_dim: int,\n        hidden: int,\n        local_cap: float,\n        step_cap: float,\n    ):\n        super().__init__()\n        self.local_cap = float(local_cap)\n        self.step_cap = float(step_cap)\n\n        self.local_net = ConvBlockND(\n            dim, latent_dim, latent_dim, hidden\n        )\n        self.local_logit = nn.Parameter(torch.tensor(-2.2))\n        self.step_logit = nn.Parameter(torch.tensor(-1.5))\n\n    def vector_field(self, psi: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        local_scale = self.local_cap * torch.sigmoid(self.local_logit)\n        raw = torch.tanh(self.local_net(psi))\n        delta_local = local_scale * raw\n\n        info = {\n            "local_scale": local_scale,\n            "local_strength": delta_local.abs().mean(),\n        }\n        return delta_local, info\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        delta_local, info = self.vector_field(psi)\n        step = self.step_cap * torch.sigmoid(self.step_logit)\n        psi_next = psi + step * delta_local\n\n        info = {\n            **info,\n            "step_size": step,\n            "delta_strength": (step * delta_local).abs().mean(),\n        }\n        return psi_next, info\n\n\nclass FieldDecoder(nn.Module):\n    def __init__(self, dim: int, latent_dim: int, channels: int, hidden: int):\n        super().__init__()\n        self.net = ConvBlockND(dim, latent_dim, channels, hidden)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass CIDMFoundationCore(nn.Module):\n    """\n    Closed Information Dynamics Model — Foundation Core v1.0.\n\n    Input:\n        history: [B, L, C, *S]\n\n    Output dict:\n        pred:       [B, K, C, *S]\n        latent0:    [B, D, *S]\n        latent:     [B, D, *S]\n        latent_seq: [B, K, D, *S]\n        diagnostics\n    """\n\n    VERSION = "CIDM-Foundation-Core-v1.0"\n\n    def __init__(self, config: CIDMFoundationConfig):\n        super().__init__()\n        self.config = config\n\n        d = config.spatial_dim\n        D = config.latent_dim\n        H = config.hidden\n\n        self.space_encoder = SpaceEncoder(\n            d, config.channels, D, H\n        )\n        self.time_encoder = TimeHistoryEncoder(\n            d, config.channels, config.hist_len, D, H\n        )\n        self.fusion = InformationStateFusion(d, D)\n\n        self.dynamics = BoundedLocalResidualDynamics(\n            d, D, H,\n            local_cap=config.local_cap,\n            step_cap=config.step_cap,\n        )\n        self.decoder = FieldDecoder(\n            d, D, config.channels, H\n        )\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        z_space = self.space_encoder(last)\n        z_time = self.time_encoder(history)\n        return self.fusion(z_space, z_time)\n\n    def step(\n        self,\n        psi: torch.Tensor,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        return self.dynamics(psi)\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi)\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        K = int(horizon or self.config.horizon)\n\n        psi0 = self.encode(history)\n        psi = psi0\n\n        preds = []\n        states = []\n        local_strengths = []\n        step_sizes = []\n        delta_strengths = []\n\n        spatial_dims = tuple(range(1, psi.ndim))\n        energy0 = (psi ** 2).mean(dim=spatial_dims)\n        energies = []\n\n        for _ in range(K):\n            psi, info = self.step(psi)\n            y = self.decode(psi)\n\n            preds.append(y)\n            states.append(psi)\n            local_strengths.append(info["local_strength"])\n            step_sizes.append(info["step_size"])\n            delta_strengths.append(info["delta_strength"])\n            energies.append((psi ** 2).mean(dim=spatial_dims))\n\n        pred = torch.stack(preds, dim=1)\n        latent_seq = torch.stack(states, dim=1)\n        energy = torch.stack(energies, dim=1)\n\n        diagnostics: Dict[str, torch.Tensor] = {\n            "local_strength": torch.stack(local_strengths).mean(),\n            "step_size": torch.stack(step_sizes).mean(),\n            "delta_strength": torch.stack(delta_strengths).mean(),\n            "latent_energy_drift": (energy[:, -1] - energy0).abs().mean(),\n        }\n\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n\n        return {\n            "pred": pred,\n            "latent0": psi0,\n            "latent": psi,\n            "latent_seq": latent_seq,\n            "diagnostics": diagnostics,\n        }\n\n\ndef build_cidm_foundation_core(\n    *,\n    spatial_dim: int,\n    channels: int,\n    hist_len: int,\n    horizon: int,\n    latent_dim: int,\n    hidden: int,\n    local_cap: float = 0.10,\n    step_cap: float = 0.30,\n) -> CIDMFoundationCore:\n    cfg = CIDMFoundationConfig(\n        spatial_dim=spatial_dim,\n        channels=channels,\n        hist_len=hist_len,\n        horizon=horizon,\n        latent_dim=latent_dim,\n        hidden=hidden,\n        local_cap=local_cap,\n        step_cap=step_cap,\n    )\n    return CIDMFoundationCore(cfg)\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n\n    # 1D smoke\n    m1 = build_cidm_foundation_core(\n        spatial_dim=1,\n        channels=1,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x1 = torch.randn(2, 4, 1, 64)\n    y1 = m1(x1)\n    assert y1["pred"].shape == (2, 6, 1, 64)\n\n    # 2D smoke\n    m2 = build_cidm_foundation_core(\n        spatial_dim=2,\n        channels=2,\n        hist_len=4,\n        horizon=6,\n        latent_dim=16,\n        hidden=32,\n    )\n    x2 = torch.randn(2, 4, 2, 24, 24)\n    y2 = m2(x2)\n    assert y2["pred"].shape == (2, 6, 2, 24, 24)\n\n    print("CIDM Foundation Core v1.0 smoke PASS")\n    print("1D params:", count_parameters(m1))\n    print("2D params:", count_parameters(m2))\n'
CIDM_FREEZE_MANIFEST = '{\n  "name": "CIDM Foundation Core",\n  "version": "v1.0 Final",\n  "freeze_basis": "A1-R1-R3 Core Component Causal Attribution & Freeze",\n  "freeze_gate": "PASS_TO_CORE_FINAL_FREEZE",\n  "default_core": {\n    "encoders": [\n      "SpaceEncoder",\n      "TimeHistoryEncoder"\n    ],\n    "fusion": "InformationStateFusion",\n    "dynamics": "BoundedLocalResidualDynamics",\n    "step": "bounded positive residual step",\n    "decoder": "FieldDecoder",\n    "rollout": "pure latent rollout",\n    "default_training": "prediction / multi-step rollout loss"\n  },\n  "evidence_summary": {\n    "NoSpaceInfo_mean_rmse_effect_pct": 4.895219760746574,\n    "NoTimeInfo_mean_rmse_effect_pct": 5.171493884445902,\n    "NoPhaseInfo_mean_rmse_effect_pct": -2.79857874424354,\n    "NoSkew_mean_rmse_effect_pct": -0.521169227349116,\n    "NoFlux_mean_rmse_effect_pct": -0.9437659673091198,\n    "NoLocal_mean_rmse_effect_pct": 2.397827861124943,\n    "BlackboxCore_mean_rmse_effect_pct": 1.7607338250780888,\n    "DirectPredictor_mean_rmse_effect_pct": 14.230675545900779,\n    "FreeChannelMix_vs_Skew_rmse_change_pct": 3.476865940431788,\n    "FreeFlux_vs_AntisymmetricFlux_rmse_change_pct": -2.5548147465622244,\n    "Sync05_vs_PredOnly_rmse_change_pct": 10.445702293374174,\n    "Sync05_sync_gain_change": 0.5611278432875664\n  },\n  "important_cautions": [\n    "DirectPredictor in A1-R1-R3 was materially smaller than Full, so its 14.23% RMSE gap is supportive rather than a parameter-matched causal estimate.",\n    "Only two seeds were used; several component effects had high seed variance.",\n    "Space/Time encoders show cross-task value but Diffusion-Reaction did not consistently require them.",\n    "The numerical cap values are architectural hyperparameters, not physical constants."\n  ],\n  "optional_components": {\n    "PhaseSpectralEncoder": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "NoPhaseInfo improved mean RMSE by 2.80%; benefits were inconsistent across tasks/seeds."\n    },\n    "SkewSymmetricChannelRotation": {\n      "status": "SPECIALIZED OPTIONAL",\n      "reason": "Removing skew slightly improved mean RMSE; however skew-constrained mixing was better than free channel mixing when a mixing branch was present."\n    },\n    "AntisymmetricFlux": {\n      "status": "SPECIALIZED CONSERVATIVE-EXCHANGE OPTIONAL",\n      "reason": "NoFlux slightly improved mean RMSE and FreeFlux outperformed antisymmetric flux on average."\n    },\n    "HiddenReservoir": {\n      "status": "OPTIONAL",\n      "reason": "Use for unresolved memory/internal closure; not part of the R3 universal core."\n    },\n    "NativeCausalPort": {\n      "status": "OPTIONAL",\n      "reason": "Use only when known causal forcing is available."\n    },\n    "Transport": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for displacement/coherent-structure dominated systems."\n    },\n    "CrossScaleTransfer": {\n      "status": "EXPERIMENTAL OPTIONAL",\n      "reason": "Use for tasks with explicit cross-scale transfer."\n    },\n    "StateSyncRegularizer": {\n      "status": "TRAINING OPTIONAL",\n      "reason": "Improved sync metric but increased mean RMSE by 10.45% in R3."\n    },\n    "CayleySkewUpdate": {\n      "status": "NUMERICAL OPTIONAL",\n      "reason": "Strictly orthogonal discretization if a skew branch is deliberately enabled."\n    },\n    "BER_SOC": {\n      "status": "SPECIALIZED EXTENSION FAMILY",\n      "reason": "Keep outside Foundation Core; appropriate for open/physical closure studies."\n    }\n  }\n}'
CIDM_COMPLETE_ARCHIVE_SOURCE = '"""\nCIDM architecture archive\n=========================\n\nReference PyTorch implementation for the finalized model family:\n\n1. CIDM / Closed Information Dynamics Model\n   - General-purpose closed information dynamics backbone.\n   - Corresponds conceptually to the previous experimental `sticm_v3_full`.\n\n2. CIDM-BER / CIDM with Boundary Energy Reservoir\n   - Physics-closure enhanced variant.\n   - Adds a latent Spectral Operator Core and a Boundary Energy Reservoir (BER)\n     residual closure layer.\n   - Corresponds conceptually to the previous experimental `sticm_soc_ber`.\n\nThis file is intended for architecture archiving and future experiment reuse. It\ncontains only model definitions and lightweight diagnostics; training/evaluation\nloops are intentionally kept outside this file.\n\nExpected input format\n---------------------\n`history`: Tensor[B, L, C, H, W]\n`port`: optional Tensor[B, C_port, H, W] or Tensor[B, horizon, C_port, H, W]\n`target`: optional Tensor[B, horizon, C, H, W] for diagnostics/loss construction\n\nOutput dictionary\n-----------------\n`pred`: Tensor[B, horizon, C, H, W]\n`latent`: final latent state\n`latent_seq`: Tensor[B, horizon, D, H, W]\n`diagnostics`: scalar tensors for energy drift, SOC strength, BER strength, etc.\n\nAuthoring note\n--------------\nThis is a cleaned reference architecture. Exact Colab benchmark implementations\nmay contain additional data-generation, plotting, logging, and ablation utilities.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass\nclass CIDMConfig:\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n    port_channels: int = 1\n    use_phase_encoder: bool = True\n    use_time_encoder: bool = True\n    use_hidden_reservoir: bool = True\n    use_flux: bool = True\n    use_reverse_diagnostics: bool = True\n    # CIDM-BER options\n    soc_modes1: int = 8\n    soc_modes2: int = 8\n    ber_add_scale: float = 0.05\n    ber_enabled: bool = True\n    soc_enabled: bool = True\n\n\ndef count_parameters(model: nn.Module) -> int:\n    """Return the number of trainable parameters."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef conv_block(in_ch: int, out_ch: int, hidden: int, groups: int = 1) -> nn.Sequential:\n    return nn.Sequential(\n        nn.Conv2d(in_ch, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, out_ch, 1),\n    )\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field into latent information state."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode a finite observation history as temporal context."""\n\n    def __init__(self, channels: int, hist_len: int, latent_dim: int):\n        super().__init__()\n        self.hist_len = hist_len\n        self.net = nn.Sequential(\n            nn.Conv2d(channels * hist_len, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: B, L, C, H, W\n        b, l, c, h, w = history.shape\n        if l != self.hist_len:\n            raise ValueError(f"Expected hist_len={self.hist_len}, got {l}.")\n        return self.net(history.reshape(b, l * c, h, w))\n\n\nclass PhaseSpectralEncoder(nn.Module):\n    """FFT-based phase/amplitude encoder projected back to the spatial latent grid."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.freq_net = nn.Sequential(\n            nn.Conv2d(channels * 3, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.spatial_net = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(x, dim=(-2, -1), norm="ortho")\n        amp = torch.log1p(torch.abs(z))\n        feats = torch.cat([z.real, z.imag, amp], dim=1)\n        f = self.freq_net(feats)\n        # Return to the spatial support while retaining frequency-aware features.\n        f_complex = torch.complex(f, torch.zeros_like(f))\n        spatial = torch.fft.ifft2(f_complex, dim=(-2, -1), norm="ortho").real\n        return self.spatial_net(spatial)\n\n\nclass FieldDecoder(nn.Module):\n    """Decode latent information state to physical/observed field."""\n\n    def __init__(self, latent_dim: int, channels: int, hidden: int = 64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, channels, 1),\n        )\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass SpectralFieldDecoder(nn.Module):\n    """Small spectral residual decoder used by CIDM for phase-aware readout."""\n\n    def __init__(self, latent_dim: int, channels: int):\n        super().__init__()\n        self.mix = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, channels, 1),\n        )\n        self.scale = nn.Parameter(torch.tensor(-2.2))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(psi, dim=(-2, -1), norm="ortho")\n        # Simple low-pass emphasis for phase-stable readout.\n        H, W = psi.shape[-2:]\n        ky = torch.fft.fftfreq(H, device=psi.device).view(1, 1, H, 1)\n        kx = torch.fft.fftfreq(W, device=psi.device).view(1, 1, 1, W)\n        filt = torch.exp(-6.0 * (kx ** 2 + ky ** 2))\n        low = torch.fft.ifft2(z * filt, dim=(-2, -1), norm="ortho").real\n        return torch.sigmoid(self.scale) * self.mix(low)\n\n\nclass AntisymmetricFlux(nn.Module):\n    """Conservative local exchange field.\n\n    For each neighbor pair, the edge update is constructed as f(a,b)-f(b,a),\n    then added to one side and subtracted from the other. This encodes the idea\n    that local exchanges can redistribute information without creating it.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.edge_net = nn.Sequential(\n            nn.Conv2d(2 * latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n\n    def edge(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\n        return self.edge_net(torch.cat([a, b], dim=1)) - self.edge_net(torch.cat([b, a], dim=1))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        out = torch.zeros_like(psi)\n        # x direction\n        right = torch.roll(psi, shifts=-1, dims=-1)\n        fx = self.edge(psi, right)\n        out = out + fx - torch.roll(fx, shifts=1, dims=-1)\n        # y direction; safe for H=1 because roll is identity and antisymmetry cancels.\n        down = torch.roll(psi, shifts=-1, dims=-2)\n        fy = self.edge(psi, down)\n        out = out + fy - torch.roll(fy, shifts=1, dims=-2)\n        return out\n\n\nclass SkewChannelRotation(nn.Module):\n    """Learned skew-symmetric channel mixing for approximately conservative flow."""\n\n    def __init__(self, latent_dim: int):\n        super().__init__()\n        self.raw = nn.Parameter(torch.randn(latent_dim, latent_dim) * 0.02)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        A = self.raw - self.raw.t()\n        return torch.einsum("ij,bjhw->bihw", A, psi)\n\n\nclass HiddenReservoirTransfer(nn.Module):\n    """Internal hidden reservoir transfer for unobserved latent exchange."""\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = conv_block(latent_dim, latent_dim, hidden)\n        self.scale = nn.Parameter(torch.tensor(-2.0))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.scale) * self.net(psi)\n\n\nclass ClosedInformationDynamicsCore(nn.Module):\n    """CIDM closed latent vector field."""\n\n    def __init__(self, latent_dim: int, hidden: int, use_flux: bool = True, use_hidden_reservoir: bool = True):\n        super().__init__()\n        self.use_flux = use_flux\n        self.use_hidden_reservoir = use_hidden_reservoir\n        self.skew = SkewChannelRotation(latent_dim)\n        self.flux = AntisymmetricFlux(latent_dim, hidden)\n        self.hidden = HiddenReservoirTransfer(latent_dim, hidden)\n        self.local = conv_block(latent_dim, latent_dim, hidden)\n        self.step_scale = nn.Parameter(torch.tensor(0.18))\n\n    def vector_field(self, psi: torch.Tensor) -> torch.Tensor:\n        dpsi = self.skew(psi)\n        if self.use_flux:\n            dpsi = dpsi + self.flux(psi)\n        if self.use_hidden_reservoir:\n            dpsi = dpsi + 0.25 * self.hidden(psi)\n        # A small local residual stabilizes non-Hamiltonian observed fields.\n        dpsi = dpsi + 0.10 * self.local(psi)\n        return dpsi\n\n    def forward(self, psi: torch.Tensor, dt_scale: float = 1.0) -> torch.Tensor:\n        alpha = torch.tanh(self.step_scale) * dt_scale\n        return psi + alpha * self.vector_field(psi)\n\n\nclass LatentSpectralOperatorCore(nn.Module):\n    """Latent-domain spectral operator for CIDM-BER.\n\n    This is a residual spectral vector field inside the latent dynamics, not an\n    external FNO baseline. It learns complex low-frequency mode mixing and maps\n    it back to the latent grid as a global correction to local closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int = 64, modes1: int = 8, modes2: int = 8):\n        super().__init__()\n        self.latent_dim = latent_dim\n        self.modes1 = modes1\n        self.modes2 = modes2\n        scale = 1.0 / max(1, latent_dim)\n        self.weight_pos = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.weight_neg = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.pre = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.post = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-1.8))\n\n    @staticmethod\n    def compl_mul2d(x: torch.Tensor, wri: torch.Tensor) -> torch.Tensor:\n        w = torch.view_as_complex(wri.contiguous())\n        return torch.einsum("bihw,iohw->bohw", x, w)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        B, D, H, W = psi.shape\n        x = self.pre(psi)\n        x_ft = torch.fft.rfft2(x, dim=(-2, -1), norm="ortho")\n        out_ft = torch.zeros(B, D, H, W // 2 + 1, dtype=torch.cfloat, device=psi.device)\n        m1 = min(self.modes1, H)\n        m2 = min(self.modes2, W // 2 + 1)\n        out_ft[:, :, :m1, :m2] = self.compl_mul2d(x_ft[:, :, :m1, :m2], self.weight_pos[:, :, :m1, :m2])\n        if H > 1:\n            out_ft[:, :, -m1:, :m2] = self.compl_mul2d(x_ft[:, :, -m1:, :m2], self.weight_neg[:, :, :m1, :m2])\n        y = torch.fft.irfft2(out_ft, s=(H, W), dim=(-2, -1), norm="ortho")\n        return torch.sigmoid(self.logit_scale) * self.post(y)\n\n\nclass BoundaryEnergyReservoir(nn.Module):\n    """Boundary Energy Reservoir (BER) residual closure layer.\n\n    BER is a virtual external energy reservoir, not a full external-state model.\n    It learns a minimal residual exchange field that accounts for open-system\n    effects not captured by the internal closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, port_channels: int, hidden: int):\n        super().__init__()\n        self.port_channels = max(1, port_channels)\n        in_ch = latent_dim * 2 + self.port_channels\n        self.net = nn.Sequential(\n            nn.Conv2d(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 3, padding=1),\n        )\n        mask_hidden = max(8, hidden // 2)\n        self.mask = nn.Sequential(\n            nn.Conv2d(in_ch, mask_hidden, 3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(mask_hidden, 1, 1),\n            nn.Sigmoid(),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-3.0))\n\n    def _port_to_grid(self, psi: torch.Tensor, port: Optional[torch.Tensor]) -> torch.Tensor:\n        B, _, H, W = psi.shape\n        if port is None or port.numel() == 0:\n            return torch.zeros(B, self.port_channels, H, W, device=psi.device, dtype=psi.dtype)\n        if port.dim() != 4:\n            raise ValueError("port must be Tensor[B, C_port, H, W] for one step.")\n        p = port\n        if p.shape[-2:] != (H, W):\n            p = F.interpolate(p, size=(H, W), mode="bilinear", align_corners=False)\n        if p.shape[1] < self.port_channels:\n            pad = torch.zeros(B, self.port_channels - p.shape[1], H, W, device=psi.device, dtype=psi.dtype)\n            p = torch.cat([p, pad], dim=1)\n        elif p.shape[1] > self.port_channels:\n            p = p[:, : self.port_channels]\n        return p\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n        internal_delta: torch.Tensor,\n        port: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        p = self._port_to_grid(psi, port)\n        x = torch.cat([psi, internal_delta, p], dim=1)\n        raw = torch.tanh(self.net(x))\n        mask = self.mask(x)\n        scale = torch.sigmoid(self.logit_scale)\n        ber_delta = scale * mask * raw\n        info = {\n            "ber_strength": ber_delta.abs().mean(),\n            "ber_mask_mean": mask.mean(),\n            "ber_scale": scale,\n        }\n        return ber_delta, info\n\n\nclass CIDM(nn.Module):\n    """Closed Information Dynamics Model.\n\n    General-purpose主体架构. This version corresponds to the stabilized closed\n    dynamics backbone: spatial information encoding, temporal history encoding,\n    phase-spectral feature encoding, conservative local exchange, internal hidden\n    reservoir transfer, and autoregressive latent rollout.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__()\n        self.config = config\n        D = config.latent_dim\n        H = config.hidden\n        self.space_encoder = SpaceEncoder(config.channels, D)\n        self.time_encoder = TimeHistoryEncoder(config.channels, config.hist_len, D)\n        self.phase_encoder = PhaseSpectralEncoder(config.channels, D)\n        fuse_inputs = D + (D if config.use_time_encoder else 0) + (D if config.use_phase_encoder else 0)\n        self.fuse = nn.Sequential(nn.Conv2d(fuse_inputs, D, 1), nn.SiLU(), nn.Conv2d(D, D, 1))\n        self.core = ClosedInformationDynamicsCore(D, H, use_flux=config.use_flux, use_hidden_reservoir=config.use_hidden_reservoir)\n        self.decoder = FieldDecoder(D, config.channels, H)\n        self.spectral_decoder = SpectralFieldDecoder(D, config.channels)\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        feats = [self.space_encoder(last)]\n        if self.config.use_time_encoder:\n            feats.append(self.time_encoder(history))\n        if self.config.use_phase_encoder:\n            feats.append(self.phase_encoder(last))\n        return self.fuse(torch.cat(feats, dim=1))\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi) + self.spectral_decoder(psi)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_next = self.core(psi)\n        return psi_next, {"soc_strength": torch.zeros((), device=psi.device), "ber_strength": torch.zeros((), device=psi.device)}\n\n    def _select_port_step(self, port: Optional[torch.Tensor], t: int) -> Optional[torch.Tensor]:\n        if port is None:\n            return None\n        if port.dim() == 5:\n            return port[:, min(t, port.shape[1] - 1)]\n        return port\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        port: Optional[torch.Tensor] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        horizon = int(horizon or self.config.horizon)\n        psi0 = self.encode(history)\n        psi = psi0\n        preds = []\n        latent_seq = []\n        soc_strengths, ber_strengths = [], []\n        energy0 = (psi0 ** 2).mean(dim=(1, 2, 3))\n        energies = []\n        for t in range(horizon):\n            psi, info = self.step(psi, t, self._select_port_step(port, t))\n            y = self.decode(psi)\n            preds.append(y)\n            latent_seq.append(psi)\n            energies.append((psi ** 2).mean(dim=(1, 2, 3)))\n            soc_strengths.append(info.get("soc_strength", torch.zeros((), device=psi.device)))\n            ber_strengths.append(info.get("ber_strength", torch.zeros((), device=psi.device)))\n        pred = torch.stack(preds, dim=1)\n        latent = torch.stack(latent_seq, dim=1)\n        energy = torch.stack(energies, dim=1)\n        diagnostics = {\n            "energy_drift": (energy[:, -1] - energy0).abs().mean(),\n            "soc_strength": torch.stack([s if s.dim() else s.expand(1) for s in soc_strengths]).mean(),\n            "ber_strength": torch.stack([b if b.dim() else b.expand(1) for b in ber_strengths]).mean(),\n        }\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n        return {"pred": pred, "latent": psi, "latent_seq": latent, "diagnostics": diagnostics}\n\n\nclass CIDM_BER(CIDM):\n    """CIDM with Spectral Operator Core and Boundary Energy Reservoir.\n\n    Physics-enhanced variant for systems with salient wave/flow propagation,\n    quasi-closed dynamics, boundary exchange, or energy closure requirements.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__(config)\n        self.soc = LatentSpectralOperatorCore(\n            config.latent_dim,\n            hidden=config.hidden,\n            modes1=config.soc_modes1,\n            modes2=config.soc_modes2,\n        )\n        self.ber = BoundaryEnergyReservoir(config.latent_dim, config.port_channels, config.hidden)\n        self.ber_add_scale = float(config.ber_add_scale)\n        self.soc_enabled = bool(config.soc_enabled)\n        self.ber_enabled = bool(config.ber_enabled)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_int = self.core(psi)\n        soc_delta = self.soc(psi) if self.soc_enabled else torch.zeros_like(psi)\n        psi_int = psi_int + soc_delta\n        internal_delta = psi_int - psi\n        if self.ber_enabled:\n            ber_delta, ber_info = self.ber(psi, internal_delta.detach(), port)\n            psi_next = psi_int + self.ber_add_scale * ber_delta\n        else:\n            ber_delta = torch.zeros_like(psi)\n            ber_info = {"ber_strength": torch.zeros((), device=psi.device), "ber_mask_mean": torch.zeros((), device=psi.device)}\n            psi_next = psi_int\n        info = {\n            "soc_strength": soc_delta.abs().mean(),\n            "ber_strength": ber_info["ber_strength"],\n            "ber_mask_mean": ber_info.get("ber_mask_mean", torch.zeros((), device=psi.device)),\n            "ber_budget_proxy": (ber_delta.mean(dim=(1, 2, 3)) - internal_delta.mean(dim=(1, 2, 3))).abs().mean(),\n        }\n        return psi_next, info\n\n\ndef build_cidm(config: Optional[CIDMConfig] = None) -> CIDM:\n    return CIDM(config or CIDMConfig())\n\n\ndef build_cidm_ber(config: Optional[CIDMConfig] = None) -> CIDM_BER:\n    cfg = config or CIDMConfig()\n    return CIDM_BER(cfg)\n\n\n# Backward-compatible aliases for previous internal experiment names.\nClosedInformationDynamicsModel = CIDM\nClosedInformationDynamicsBERModel = CIDM_BER\nSTICM_V3_FULL_ALIAS = CIDM\nSTICM_SOC_BER_ALIAS = CIDM_BER\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    cfg = CIDMConfig(channels=1, hist_len=4, horizon=1, latent_dim=4, hidden=8, port_channels=1)\n    x = torch.randn(1, cfg.hist_len, cfg.channels, 4, 4)\n    port = torch.randn(1, cfg.port_channels, 4, 4)\n    for cls in [CIDM, CIDM_BER]:\n        model = cls(cfg)\n        out = model(x, port=port)\n        print(cls.__name__, out["pred"].shape, "params=", count_parameters(model), out["diagnostics"])\n'

EXPECTED_CORE_SHA256 = "e5e26b35765301e9ecd25aec5a6c507395ff370dca11c820f540ff0f1e79f510"
EXPECTED_MANIFEST_SHA256 = "9508dd000005a154d3d3fa4437602d4214a6266d630bf13af855e5517649ecbd"
EXPECTED_COMPLETE_ARCHIVE_SHA256 = "f9b1cf4ff7630f5540bb3adde20e4c4b81fff3218d9325ea3105bf28b1d97665"

core_file=FROZEN/"cidm_foundation_core_v1.py"
manifest_file=FROZEN/"CIDM_Foundation_Core_v1_0_FREEZE_MANIFEST.json"
complete_file=FROZEN/"cidm_complete_archived_topology.py"

core_file.write_text(CIDM_CORE_SOURCE,encoding="utf-8")
manifest_file.write_text(CIDM_FREEZE_MANIFEST,encoding="utf-8")
complete_file.write_text(CIDM_COMPLETE_ARCHIVE_SOURCE,encoding="utf-8")

core_sha=sha256_file(core_file)
manifest_sha=sha256_file(manifest_file)
complete_sha=sha256_file(complete_file)

print("CIDM Core SHA256:",core_sha)
print("Core Manifest SHA256:",manifest_sha)
print("CIDM-Complete archive SHA256:",complete_sha)

assert core_sha==EXPECTED_CORE_SHA256
assert manifest_sha==EXPECTED_MANIFEST_SHA256
assert complete_sha==EXPECTED_COMPLETE_ARCHIVE_SHA256

import importlib.util

spec=importlib.util.spec_from_file_location("cidm_foundation_core_v1",core_file)
cidm_mod=importlib.util.module_from_spec(spec)
sys.modules["cidm_foundation_core_v1"]=cidm_mod
spec.loader.exec_module(cidm_mod)

CIDMFoundationCore=cidm_mod.CIDMFoundationCore
CIDMFoundationConfig=cidm_mod.CIDMFoundationConfig
build_cidm_foundation_core=cidm_mod.build_cidm_foundation_core

spec2=importlib.util.spec_from_file_location("cidm_complete_archive",complete_file)
cidm_complete_mod=importlib.util.module_from_spec(spec2)
sys.modules["cidm_complete_archive"]=cidm_complete_mod
spec2.loader.exec_module(cidm_complete_mod)

CIDMCompleteConfig=cidm_complete_mod.CIDMConfig
CIDMCompleteArchived=cidm_complete_mod.CIDM_BER

# Core smoke
m=build_cidm_foundation_core(
    spatial_dim=2,channels=1,hist_len=4,horizon=6,
    latent_dim=16,hidden=32
).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 2D smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=2,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,24,24,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,24,24)
del m,x,y

# Complete 1D-via-H=1 smoke
cfg=CIDMCompleteConfig(
    channels=1,hist_len=4,horizon=6,
    latent_dim=8,hidden=16,port_channels=1,
    use_phase_encoder=True,use_time_encoder=True,
    use_hidden_reservoir=True,use_flux=True,
    soc_modes1=1,soc_modes2=4,
    ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
)
m=CIDMCompleteArchived(cfg).to(DEVICE)
x=torch.randn(2,4,1,1,64,device=DEVICE)
with torch.no_grad():
    y=m(x,horizon=6,port=None)["pred"]
assert y.shape==(2,6,1,1,64)
del m,x,y
torch.cuda.empty_cache()

COMPLETE_CONTRACT={
    "benchmark_name":"CIDM-Complete",
    "executable_mapping":"cidm_architectures.py::CIDM_BER",
    "archive_sha256":complete_sha,
    "components":[
        "SpaceEncoder","TimeHistoryEncoder","PhaseSpectralEncoder",
        "SkewChannelRotation","AntisymmetricFlux","HiddenReservoirTransfer",
        "LocalResidual","LatentSpectralOperatorCore","BoundaryEnergyReservoir",
        "FieldDecoder","SpectralFieldDecoder","PureLatentRollout"
    ],
    "known_forcing_port":"disabled / None because PDEBench headline tasks provide no common forcing field",
    "static_fields":"not supplied; absent from archived executable implementation",
    "1d_protocol":"lift to H=1, run exact archived Conv2D implementation, squeeze H=1",
    "future_target_at_inference":"forbidden",
    "note":"Figure-level State-Observation Closure is not invented here; archived executable SOC is LatentSpectralOperatorCore."
}
save_json(COMPLETE_CONTRACT,OUT/"CIDM_Complete_Benchmark_Contract.json")

REPORT["cidm_freeze"]={
    "core_sha256":core_sha,
    "manifest_sha256":manifest_sha,
    "exact":True,
}
REPORT["cidm_complete_freeze"]={
    "archive_sha256":complete_sha,
    "exact":True,
    "contract":COMPLETE_CONTRACT,
}

print("FROZEN CIDM CORE: PASS")
print("FROZEN CIDM-COMPLETE ARCHIVE: PASS")


## 2. Exact official source provenance

In [ ]:

# ============================================================
# 2. Exact source checkout + official imports
# ============================================================
LOCKED={
    "pdebench":{
        "url":"https://github.com/pdebench/PDEBench.git",
        "sha":"4ff3e3a4aa1561721b5571fa3a048a0a463e0568",
    },
    "neuraloperator":{
        "url":"https://github.com/neuraloperator/neuraloperator.git",
        "sha":"00b7d86f8d74ff0af55da53eb585fe26df9c71f0",
    },
    "transolver":{
        "url":"https://github.com/thuml/Transolver.git",
        "sha":"75e0f67643806a81cd1d3f6adc88dd8c02416fe7",
    },
}
checkout=[]
for rid,m in LOCKED.items():
    dst=SRC/rid
    if dst.exists():
        rc,head=run(["git","rev-parse","HEAD"],cwd=dst,timeout=60)
        if rc==0 and head.strip()==m["sha"]:
            print("[reuse source]",rid,head.strip())
            checkout.append({"id":rid,"head":head.strip(),"exact":True})
            continue
        shutil.rmtree(dst)

    rc,o=run(["git","clone","--filter=blob:none","--no-checkout",m["url"],str(dst)])
    if rc: raise RuntimeError(f"{rid} clone failed\n{o[-3000:]}")
    rc,o=run(["git","fetch","origin",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} fetch failed\n{o[-3000:]}")
    rc,o=run(["git","checkout","--detach",m["sha"]],cwd=dst)
    if rc: raise RuntimeError(f"{rid} checkout failed\n{o[-3000:]}")
    _,head=run(["git","rev-parse","HEAD"],cwd=dst)
    head=head.strip()
    checkout.append({"id":rid,"head":head,"exact":head==m["sha"]})
    print(rid,head,"PASS" if head==m["sha"] else "FAIL")

assert all(x["exact"] for x in checkout)
pd.DataFrame(checkout).to_csv(OUT/"source_checkout.csv",index=False)

# Dependencies needed by the exact official cores.
rc,o=run([
    sys.executable,"-m","pip","install","-q",
    "tensorly","tensorly-torch","einops","timm","scipy"
],timeout=1200)
if rc:
    raise RuntimeError(o[-5000:])

sys.path.insert(0,str(SRC/"pdebench"))
sys.path.insert(0,str(SRC/"neuraloperator"))
TRANS_BENCH=SRC/"transolver"/"PDE-Solving-StandardBenchmark"
sys.path.insert(0,str(TRANS_BENCH))

from pdebench.models.fno.fno import FNO1d as PDE_FNO1d, FNO2d as PDE_FNO2d
from pdebench.models.unet.unet import UNet1d as PDE_UNet1d, UNet2d as PDE_UNet2d
from neuralop.models import FNO as NeuralOpFNO
from model.Transolver_Irregular_Mesh import Model as TransolverIrregular
from model.Transolver_Structured_Mesh_2D import Model as Transolver2D

print("OFFICIAL IMPORTS: PASS")
REPORT["source_checkout"]=checkout


## 3. Frozen/official wrappers

In [ ]:

# ============================================================
# 4. Official wrappers
# ============================================================
def normalized_grid(spatial,device):
    if len(spatial)==1:
        return torch.linspace(0,1,spatial[0],device=device)[:,None]
    x=torch.linspace(0,1,spatial[0],device=device)
    y=torch.linspace(0,1,spatial[1],device=device)
    X,Y=torch.meshgrid(x,y,indexing="ij")
    return torch.stack([X,Y],-1)

class CIDMWrapper(nn.Module):
    def __init__(self,dim,c,L,K,D,H):
        super().__init__()
        self.model=build_cidm_foundation_core(
            spatial_dim=dim,channels=c,hist_len=L,horizon=K,
            latent_dim=D,hidden=H
        )
    def rollout(self,h,K):
        return self.model(h,horizon=K)["pred"]


class CIDMCompleteWrapper(nn.Module):
    """
    Exact archived CIDM_BER topology used as CIDM-Complete.
    For 1D tasks we lift X -> [1,X] and use the same archived Conv2D model.
    No external forcing/port is supplied in the PDEBench headline panel.
    """
    def __init__(self,dim,c,L,K,D,H,modes1,modes2):
        super().__init__()
        self.dim=dim
        self.config=CIDMCompleteConfig(
            channels=c,
            hist_len=L,
            horizon=K,
            latent_dim=D,
            hidden=H,
            port_channels=1,
            use_phase_encoder=True,
            use_time_encoder=True,
            use_hidden_reservoir=True,
            use_flux=True,
            use_reverse_diagnostics=True,
            soc_modes1=modes1,
            soc_modes2=modes2,
            ber_add_scale=0.05,
            ber_enabled=True,
            soc_enabled=True,
        )
        self.model=CIDMCompleteArchived(self.config)

    def _lift(self,h):
        return h.unsqueeze(-2) if self.dim==1 else h

    def rollout(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        p=out["pred"]
        return p.squeeze(-2) if self.dim==1 else p

    @torch.no_grad()
    def architecture_diagnostics(self,h,K):
        hh=self._lift(h)
        out=self.model(hh,horizon=K,port=None)
        d=out["diagnostics"]
        return {
            "latent_energy_drift":float(d["energy_drift"].detach()),
            "soc_strength":float(d["soc_strength"].detach()),
            "ber_strength":float(d["ber_strength"].detach()),
        }


class PDEBenchFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        if dim==1:
            self.model=PDE_FNO1d(c,modes=modes,width=width,initial_step=L)
        else:
            self.model=PDE_FNO2d(c,modes1=modes,modes2=modes,width=width,initial_step=L)
    def one(self,h):
        B,L,C,*S=h.shape
        if self.dim==1:
            inp=h.permute(0,3,1,2).reshape(B,S[0],L*C)
        else:
            inp=h.permute(0,3,4,1,2).reshape(B,S[0],S[1],L*C)
        grid=normalized_grid(S,h.device)[None].expand(B,*([-1]*self.dim),-1)
        y=self.model(inp,grid).squeeze(-2)
        if self.dim==1: return y.permute(0,2,1).contiguous()
        return y.permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class PDEBenchUNetWrapper(nn.Module):
    def __init__(self,dim,c,L,features):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=(PDE_UNet1d if dim==1 else PDE_UNet2d)(
            in_channels=c*L,out_channels=c,init_features=features
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class NeuralOpFNOWrapper(nn.Module):
    def __init__(self,dim,c,L,width,modes):
        super().__init__(); self.dim=dim; self.c=c; self.L=L
        self.model=NeuralOpFNO(
            n_modes=tuple([modes]*dim),
            in_channels=c*L,out_channels=c,
            hidden_channels=width,n_layers=4,
            positional_embedding="grid",
            fno_block_precision="full",
        )
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            B,L,C,*S=h.shape
            y=self.model(h.reshape(B,L*C,*S))
            ps.append(y); h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)

class TransolverWrapper(nn.Module):
    def __init__(self,dim,c,L,S,n_hidden,n_layers,n_head,slice_num):
        super().__init__(); self.dim=dim; self.c=c; self.L=L; self.S=S
        kw=dict(
            space_dim=dim,n_layers=n_layers,n_hidden=n_hidden,dropout=0.0,
            n_head=n_head,Time_Input=False,act="gelu",mlp_ratio=1,
            fun_dim=c*L,out_dim=c,slice_num=slice_num,unified_pos=False
        )
        if dim==1:
            self.model=TransolverIrregular(**kw)
        else:
            self.model=Transolver2D(**kw,H=S[0],W=S[1])
    def one(self,h):
        B,L,C,*S=h.shape
        g=normalized_grid(S,h.device).reshape(-1,self.dim)[None].expand(B,-1,-1)
        if self.dim==1:
            fx=h.permute(0,3,1,2).reshape(B,-1,L*C)
        else:
            fx=h.permute(0,3,4,1,2).reshape(B,-1,L*C)
        y=self.model(g,fx)
        if self.dim==1:
            return y.reshape(B,S[0],C).permute(0,2,1).contiguous()
        return y.reshape(B,S[0],S[1],C).permute(0,3,1,2).contiguous()
    def rollout(self,h,K):
        ps=[]
        for _ in range(K):
            y=self.one(h); ps.append(y)
            h=torch.cat([h[:,1:],y[:,None]],1)
        return torch.stack(ps,1)



## 4. HF Dataset restore / per-run sync

本 FIX 使用两个 HF prefix：

```text
Base A2 (read-only):
A2_BroadDynamics_500K_3Seed/

Repair experiment:
A2_R1_DiffSorp1D_FIX_500K_3Seed/
```

启动时：
- 恢复本 FIX 的 result/history/checkpoint；
- 搜索 Base A2 的 120 个 result JSON；
- 搜索 Base A2 的旧 Diff-Sorp compact cache；
- 不覆盖旧 A2 原始结果。


In [ ]:
# ============================================================
# 4. HF DATASET persistent runtime + base-A2 restore
# ============================================================
HF_DATASET_REPO_ID=os.getenv(
    "CIDM_HF_DATASET_REPO",
    "wuff-mann/CIDM-v1"
).strip()
HF_REPO_TYPE="dataset"
HF_BASE_PREFIX="A2_BroadDynamics_500K_3Seed"
HF_FIX_PREFIX="A2_R1_DiffSorp1D_FIX_500K_3Seed"
HF_CHECKPOINT_EVERY=int(os.getenv("CIDM_HF_CHECKPOINT_EVERY","500"))
HF_MAX_RETRIES=int(os.getenv("CIDM_HF_MAX_RETRIES","4"))
HF_ALLOW_LOCAL_ONLY=os.getenv("CIDM_HF_ALLOW_LOCAL_ONLY","0")=="1"

try:
    from google.colab import userdata
    HF_TOKEN_RESOLVED=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN_RESOLVED=os.getenv("HF_TOKEN")

HF_ENABLED=False
HF_API=None
HF_REMOTE_FILES=[]

def _utc_now():
    return datetime.now(timezone.utc).isoformat()

def _retry_call(fn,*args,**kwargs):
    last=None
    for attempt in range(1,HF_MAX_RETRIES+1):
        try:
            return fn(*args,**kwargs)
        except Exception as e:
            last=e
            if attempt==HF_MAX_RETRIES:
                break
            wait=min(30,2**attempt)
            print(f"[HF retry {attempt}] {type(e).__name__}: {e}; sleep {wait}s")
            time.sleep(wait)
    raise last

def scan_fix_progress():
    rows=[]
    for p in sorted(OUT.glob("A2R1__*_result.json")):
        try:
            rows.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception:
            pass
    return {
        "expected":len(SEEDS)*len(MODELS),
        "results":len(rows),
        "completed":sum(x.get("status")=="COMPLETED" for x in rows),
        "checkpoints":len(list(CKPT.glob("A2R1__*_latest.pt"))),
    }

def write_fix_manifest(run_id=None,step=None,event=None):
    state=scan_fix_progress()
    state.update({
        "repo_id":HF_DATASET_REPO_ID,
        "repo_type":"dataset",
        "prefix":HF_FIX_PREFIX,
        "updated_at_utc":_utc_now(),
        "current_run":run_id,
        "current_step":step,
        "event":event,
    })
    p=OUT/"A2_R1_progress_manifest.json"
    p.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding="utf-8")
    rows=[]
    for q in sorted(OUT.glob("A2R1__*_result.json")):
        try:
            rows.append(json.loads(q.read_text(encoding="utf-8")))
        except Exception:
            pass
    if rows:
        pd.DataFrame(rows).to_csv(OUT/"A2_R1_run_results_partial.csv",index=False)
    return p,state

def _fix_repo_path(local_path):
    p=Path(local_path).resolve()
    rel=p.relative_to(OUT.resolve()).as_posix()
    return f"{HF_FIX_PREFIX}/{rel}"

def _hf_commit_fix(files,message):
    if not HF_ENABLED:
        return
    from huggingface_hub import CommitOperationAdd
    ops=[]
    for p in files:
        p=Path(p)
        if p.exists() and p.is_file():
            ops.append(CommitOperationAdd(
                path_in_repo=_fix_repo_path(p),
                path_or_fileobj=str(p)
            ))
    if ops:
        _retry_call(
            HF_API.create_commit,
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            operations=ops,
            commit_message=message,
        )

def hf_sync_checkpoint(path,run_id,step):
    if not HF_ENABLED:
        return
    man,_=write_fix_manifest(run_id,step,"MID_RUN_CHECKPOINT")
    try:
        _hf_commit_fix([path,man],f"[A2-R1] checkpoint {run_id} step={step}")
        print(f"[HF A2-R1 CHECKPOINT] {run_id} step={step}")
    except Exception as e:
        print("[HF A2-R1 checkpoint warning]",e)

def hf_sync_finished(run_id,result_path,history_path,latest_path):
    if not HF_ENABLED:
        return
    row=json.loads(Path(result_path).read_text(encoding="utf-8"))
    status=row.get("status","UNKNOWN")
    man,state=write_fix_manifest(run_id,None,f"RUN_{status}")
    files=[
        result_path,history_path,man,
        OUT/"A2_R1_run_results_partial.csv"
    ]
    try:
        parts=run_id.split("__")
        model=parts[1]; seed=int(parts[3][1:])
        if seed==SEEDS[0]:
            q=PRED/f"diff_sorp_1d__{model}.pt"
            if q.exists():
                files.append(q)
    except Exception:
        pass
    if status!="COMPLETED" and Path(latest_path).exists():
        files.append(latest_path)

    try:
        _hf_commit_fix(
            files,
            f"[A2-R1] {status} {run_id} completed={state['completed']}/{state['expected']}"
        )
        print(f"[HF A2-R1 RUN SYNC] {run_id}: {status} {state['completed']}/{state['expected']}")
        if status=="COMPLETED":
            try:
                HF_API.delete_file(
                    path_in_repo=_fix_repo_path(latest_path),
                    repo_id=HF_DATASET_REPO_ID,
                    repo_type="dataset",
                    commit_message=f"[A2-R1] cleanup {run_id} checkpoint",
                )
            except Exception:
                pass
    except Exception as e:
        print("[HF A2-R1 run sync warning]",e)

def hf_download(remote,local_root="/content/_cidm_a2r1_restore"):
    from huggingface_hub import hf_hub_download
    return Path(_retry_call(
        hf_hub_download,
        repo_id=HF_DATASET_REPO_ID,
        filename=remote,
        repo_type="dataset",
        revision="main",
        token=HF_TOKEN_RESOLVED,
        local_dir=local_root,
    ))

def restore_fix_progress():
    fix_files=[f for f in HF_REMOTE_FILES if f.startswith(HF_FIX_PREFIX+"/")]
    for rf in fix_files:
        name=Path(rf).name
        if name.startswith("A2R1__") and name.endswith("_result.json"):
            dst=OUT/name
        elif name.startswith("A2R1__") and name.endswith("_history.csv"):
            dst=OUT/name
        elif name.startswith("A2R1__") and name.endswith("_latest.pt"):
            dst=CKPT/name
        elif name=="A2_R1_progress_manifest.json":
            dst=OUT/name
        else:
            continue
        try:
            src=hf_download(rf)
            dst.parent.mkdir(parents=True,exist_ok=True)
            if dst.exists() and dst.name.endswith("_result.json"):
                local=json.loads(dst.read_text(encoding="utf-8"))
                if local.get("status")=="COMPLETED":
                    continue
            if dst.exists() and dst.name.endswith("_latest.pt"):
                try:
                    l=torch.load(dst,map_location="cpu",weights_only=False)
                    r=torch.load(src,map_location="cpu",weights_only=False)
                    if int(l.get("step",-1))>=int(r.get("step",-1)):
                        continue
                except Exception:
                    pass
            shutil.copy2(src,dst)
        except Exception as e:
            print("[restore fix warning]",rf,e)
    print("[A2-R1 restore]",scan_fix_progress())

def restore_base_a2_results():
    result_files=[
        f for f in HF_REMOTE_FILES
        if f.startswith(HF_BASE_PREFIX+"/")
        and Path(f).name.startswith("A2__")
        and Path(f).name.endswith("_result.json")
    ]
    print("[Base A2 remote result JSONs]",len(result_files))
    for rf in result_files:
        dst=BASE_A2_RESULTS/Path(rf).name
        if dst.exists():
            continue
        try:
            src=hf_download(rf)
            shutil.copy2(src,dst)
        except Exception as e:
            print("[base result restore warning]",rf,e)

    base=[]
    for p in BASE_A2_RESULTS.glob("A2__*_result.json"):
        try:
            x=json.loads(p.read_text(encoding="utf-8"))
            if x.get("status")=="COMPLETED":
                base.append(x)
        except Exception:
            pass
    print("[Base A2 completed restored]",len(base),"/120")
    return base

def locate_old_diffsorp_cache():
    candidates=[
        f for f in HF_REMOTE_FILES
        if f.startswith(HF_BASE_PREFIX+"/cache/")
        and "diff_sorp_1d_L4_K6" in Path(f).name
        and f.endswith(".pt")
    ]
    if not candidates:
        return None
    # Prefer exact A2 production cache name if duplicates exist.
    candidates=sorted(candidates,key=lambda x:(len(x),x))
    rf=candidates[0]
    local=CACHE_ROOT/("A2_original_invalid_"+Path(rf).name)
    if not local.exists():
        src=hf_download(rf)
        shutil.copy2(src,local)
    print("[old A2 diff-sorp cache restored]",rf)
    return local

# Init
if HF_TOKEN_RESOLVED:
    subprocess.run(
        [sys.executable,"-m","pip","install","-q","-U","huggingface_hub"],
        check=True
    )
    from huggingface_hub import HfApi
    HF_API=HfApi(token=HF_TOKEN_RESOLVED)
    who=_retry_call(HF_API.whoami)
    print("[HF AUTH]",who.get("name"))
    _retry_call(
        HF_API.dataset_info,
        repo_id=HF_DATASET_REPO_ID,
        revision="main"
    )
    HF_ENABLED=True
    HF_REMOTE_FILES=_retry_call(
        HF_API.list_repo_files,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        revision="main"
    )
    print("[HF DATASET VERIFIED]",HF_DATASET_REPO_ID)
    restore_fix_progress()
    BASE_A2_ROWS=restore_base_a2_results()
    OLD_DIFFSORP_CACHE=locate_old_diffsorp_cache()
else:
    if not HF_ALLOW_LOCAL_ONLY:
        raise RuntimeError("HF_TOKEN required for A2-R1 repair/merge.")
    BASE_A2_ROWS=[]
    OLD_DIFFSORP_CACHE=None



## 5. Correct Diffusion–Sorption cache

核心审计不是“能运行”，而是必须验证：

```text
history.ndim == 4        # [N,L,C,X]
target.ndim  == 4        # [N,K,C,X]
channels     == 1
spatial      == [256]
```

同时保存 before/after contract evidence。


In [ ]:

# ============================================================
# 5. Diff-Sorp 1D cache repair
# ============================================================
TASK="diff_sorp_1d"
M=TASKS[TASK]
FIX_CACHE=CACHE_ROOT/"diff_sorp_1d_CORRECT_1D_L4_K6_tr128_va32_te32_w2_s4.pt"

def repair_old_cache(old_path):
    old=torch.load(old_path,map_location="cpu",weights_only=False)
    out=copy.deepcopy(old)

    audit={"source":"A2 old compact cache","before":{},"after":{}}

    for split in ["train","val","test"]:
        h=old["splits"][split]["history"]
        y=old["splits"][split]["target"]
        audit["before"][split]={
            "history_shape":list(h.shape),
            "target_shape":list(y.shape),
        }

        # Exact known A2 error: [N,L,1,X,1] -> [N,L,1,X]
        if h.ndim==5 and h.shape[-1]==1:
            h=h.squeeze(-1)
        if y.ndim==5 and y.shape[-1]==1:
            y=y.squeeze(-1)

        out["splits"][split]["history"]=h.contiguous()
        out["splits"][split]["target"]=y.contiguous()

        audit["after"][split]={
            "history_shape":list(h.shape),
            "target_shape":list(y.shape),
        }

    out["repair_version"]="A2-R1-FIX1"
    out["repair_note"]="Removed erroneous terminal singleton spatial axis from official [T,X,1] Diff-Sorp data."
    out["original_a2_cache"]=Path(old_path).name
    return out,audit

def resume_download(url,path,expected_md5,chunk=8*1024*1024):
    path=Path(path)
    if path.exists() and md5_file(path)==expected_md5:
        return
    if path.exists():
        path.unlink()
    part=path.with_suffix(path.suffix+".part")
    pos=part.stat().st_size if part.exists() else 0
    headers={"Range":f"bytes={pos}-"} if pos else {}
    with requests.get(url,stream=True,headers=headers,timeout=(30,300)) as r:
        if pos and r.status_code==200:
            part.unlink(missing_ok=True); pos=0
        r.raise_for_status()
        mode="ab" if pos and r.status_code==206 else "wb"
        with open(part,mode) as f:
            for b in r.iter_content(chunk_size=chunk):
                if b:
                    f.write(b)
    part.replace(path)
    got=md5_file(path)
    if got!=expected_md5:
        raise RuntimeError(f"MD5 mismatch: {got}")

def build_from_raw():
    raw=Path("/content/CIDM_PDEBench_Data")/M["filename"]
    raw.parent.mkdir(parents=True,exist_ok=True)
    resume_download(M["url"],raw,M["md5"])

    with h5py.File(raw,"r") as f:
        groups=[
            k for k in sorted(f.keys())
            if isinstance(f[k],h5py.Group) and "data" in f[k]
        ]
        N=len(groups)
        if not groups:
            raise RuntimeError("Official Diff-Sorp grouped schema not found.")

        # Verify official raw trajectory contract.
        probe=np.asarray(f[groups[0]]["data"])
        if probe.ndim!=3 or probe.shape[-1]!=1:
            raise RuntimeError(
                f"Unexpected official Diff-Sorp raw shape: {probe.shape}; "
                "expected [T,X,1]."
            )

        nt=max(1,int(.10*N))
        test_pool=np.arange(0,nt)
        trainval=np.arange(nt,N)
        nv=max(1,int(.10*N))
        val_pool=trainval[-nv:]
        train_pool=trainval[:-nv]

        def pick(pool,n):
            n=min(n,len(pool))
            return pool[np.linspace(0,len(pool)-1,n,dtype=int)].tolist()

        ids={
            "train":pick(train_pool,128),
            "val":pick(val_pool,32),
            "test":pick(test_pool,32),
        }

        out={
            "task":TASK,
            "family":M["family"],
            "source_filename":M["filename"],
            "source_md5":M["md5"],
            "schema":"grouped_[T,X,1]_corrected_to_1d",
            "history":HISTORY,
            "horizon":HORIZON,
            "spatial_stride":4,
            "selected_trajectory_ids":ids,
            "splits":{},
            "repair_version":"A2-R1-FIX1-RAW",
        }

        for split,idxs in ids.items():
            HH=[]; YY=[]; WM=[]
            for idx in idxs:
                a=np.asarray(f[groups[idx]]["data"],dtype=np.float32)
                if a.ndim!=3 or a.shape[-1]!=1:
                    raise RuntimeError((groups[idx],a.shape))
                # [T,X,1] -> [T,X] -> [T,1,X]
                a=a[...,0]
                a=a[:,None,::4]
                tr=torch.from_numpy(np.ascontiguousarray(a))

                mx=tr.shape[0]-HISTORY-HORIZON
                starts=np.unique(np.linspace(0,mx,2,dtype=int))
                for st in starts:
                    HH.append(tr[st:st+HISTORY])
                    YY.append(tr[st+HISTORY:st+HISTORY+HORIZON])
                    WM.append((int(idx),int(st)))

            out["splits"][split]={
                "history":torch.stack(HH).contiguous(),
                "target":torch.stack(YY).contiguous(),
                "window_meta":WM,
            }

    # stats
    h=out["splits"]["train"]["history"].double()
    y=out["splits"]["train"]["target"].double()
    vals=torch.cat([
        h.permute(2,0,1,3).reshape(1,-1),
        y.permute(2,0,1,3).reshape(1,-1),
    ],dim=1)
    out["norm"]={
        "mean":vals.mean(1).float(),
        "std":vals.std(1,unbiased=False).clamp_min(1e-6).float(),
    }
    return out,{
        "source":"official raw DaRUS",
        "raw_probe_shape":list(probe.shape),
    }

if FIX_CACHE.exists():
    C=torch.load(FIX_CACHE,map_location="cpu",weights_only=False)
    repair_audit={"source":"existing FIX cache"}
elif OLD_DIFFSORP_CACHE is not None and Path(OLD_DIFFSORP_CACHE).exists():
    C,repair_audit=repair_old_cache(OLD_DIFFSORP_CACHE)
    torch.save(C,FIX_CACHE)
else:
    C,repair_audit=build_from_raw()
    torch.save(C,FIX_CACHE)

# Hard contract checks.
for split in ["train","val","test"]:
    h=C["splits"][split]["history"]
    y=C["splits"][split]["target"]
    assert h.ndim==4,(split,h.shape)
    assert y.ndim==4,(split,y.shape)
    assert h.shape[2]==1 and y.shape[2]==1
    assert list(h.shape[3:])==[256],h.shape
    assert list(y.shape[3:])==[256],y.shape
    assert torch.isfinite(h).all() and torch.isfinite(y).all()

# If repaired from old cache, normalization should already be identical,
# but recompute from actual corrected tensors to make the contract explicit.
h=C["splits"]["train"]["history"].double()
y=C["splits"]["train"]["target"].double()
allv=torch.cat([
    h.permute(2,0,1,3).reshape(1,-1),
    y.permute(2,0,1,3).reshape(1,-1),
],dim=1)
C["norm"]={
    "mean":allv.mean(1).float(),
    "std":allv.std(1,unbiased=False).clamp_min(1e-6).float(),
}
torch.save(C,FIX_CACHE)

repair_audit["final"]={
    "history_shape":list(C["splits"]["train"]["history"].shape),
    "target_shape":list(C["splits"]["train"]["target"].shape),
    "dim":1,
    "channels":1,
    "spatial":[256],
    "norm_mean":C["norm"]["mean"].tolist(),
    "norm_std":C["norm"]["std"].tolist(),
}
save_json(repair_audit,OUT/"A2_R1_data_contract_repair_audit.json")

# Upload corrected compact cache once.
if HF_ENABLED:
    from huggingface_hub import CommitOperationAdd
    try:
        HF_API.create_commit(
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            operations=[CommitOperationAdd(
                path_in_repo=f"{HF_FIX_PREFIX}/cache/{FIX_CACHE.name}",
                path_or_fileobj=str(FIX_CACHE)
            )],
            commit_message="[A2-R1] corrected Diff-Sorp 1D compact cache",
        )
    except Exception as e:
        print("[cache upload warning]",e)

CACHES={TASK:C}

class ChannelNormalizer:
    def __init__(self,mean,std):
        self.mean=mean.float()
        self.std=std.float().clamp_min(1e-6)
    def denorm_target(self,x):
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return x*self.std.view(*shape).to(x.device)+self.mean.view(*shape).to(x.device)

NORMALIZERS={
    TASK:ChannelNormalizer(C["norm"]["mean"],C["norm"]["std"])
}

class FrozenWindows(Dataset):
    def __init__(self,c,split,norm):
        self.h=c["splits"][split]["history"].float()
        self.y=c["splits"][split]["target"].float()
        self.norm=norm
    def __len__(self):
        return len(self.h)
    def __getitem__(self,i):
        h=self.h[i]; y=self.y[i]
        mh=self.norm.mean.view(1,-1,*([1]*(h.ndim-2)))
        sh=self.norm.std.view(1,-1,*([1]*(h.ndim-2)))
        return (h-mh)/sh,(y-mh)/sh

def loaders_for(task,batch,seed):
    c=CACHES[task]; norm=NORMALIZERS[task]
    g=torch.Generator().manual_seed(seed)
    return (
        DataLoader(FrozenWindows(c,"train",norm),batch_size=batch,shuffle=True,
                   generator=g,num_workers=0,drop_last=True,pin_memory=True),
        DataLoader(FrozenWindows(c,"val",norm),batch_size=batch,shuffle=False,
                   num_workers=0,pin_memory=True),
        DataLoader(FrozenWindows(c,"test",norm),batch_size=batch,shuffle=False,
                   num_workers=0,pin_memory=True),
    )

def task_spec(task):
    h=CACHES[task]["splits"]["train"]["history"]
    return h.ndim-3,h.shape[2],tuple(h.shape[3:])

dim,ch,sp=task_spec(TASK)
assert dim==1 and ch==1 and list(sp)==[256]
print("CORRECTED DIFF-SORP CONTRACT: PASS",dim,ch,sp)
display(pd.DataFrame([{
    "task":TASK,"dim":dim,"channels":ch,"spatial":list(sp),
    "train_windows":len(C["splits"]["train"]["history"]),
    "val_windows":len(C["splits"]["val"]["history"]),
    "test_windows":len(C["splits"]["test"]["history"]),
    "norm_mean":C["norm"]["mean"].tolist(),
    "norm_std":C["norm"]["std"].tolist(),
}]))


## 6. Correct 1D 500K parameter matching

In [ ]:

# ============================================================
# 5. A2 automatic parameter-only matching
# ============================================================
def build_by_config(task,name,cfg):
    dim,c,S=task_spec(task)

    if name=="CIDM-Foundation-v1":
        return CIDMWrapper(
            dim,c,HISTORY,HORIZON,
            cfg["latent_dim"],cfg["hidden"]
        )

    if name=="CIDM-Complete":
        return CIDMCompleteWrapper(
            dim,c,HISTORY,HORIZON,
            cfg["latent_dim"],cfg["hidden"],
            cfg["soc_modes1"],cfg["soc_modes2"]
        )

    if name=="PDEBench-FNO":
        return PDEBenchFNOWrapper(
            dim,c,HISTORY,cfg["width"],cfg["modes"]
        )

    if name=="NeuralOperator-FNO":
        return NeuralOpFNOWrapper(
            dim,c,HISTORY,cfg["width"],cfg["modes"]
        )

    if name=="Transolver":
        return TransolverWrapper(
            dim,c,HISTORY,S,
            cfg["n_hidden"],cfg["n_layers"],
            cfg["n_head"],cfg["slice_num"]
        )

    raise KeyError(name)

def search_cidm_core(task):
    dim,c,S=task_spec(task)
    cands=[]
    for D in range(16,145,4):
        hs=sorted(set([
            max(16,D),
            max(16,int(round(1.5*D/8)*8)),
            max(16,2*D),
            max(16,int(round(2.5*D/8)*8)),
            max(16,3*D),
        ]))
        for H in hs:
            m=CIDMWrapper(dim,c,HISTORY,HORIZON,D,H)
            p=count_params(m)
            cands.append((abs(p-TARGET_PARAMS),p,{
                "latent_dim":D,"hidden":H
            }))
            del m
    return min(cands,key=lambda x:x[0])

def complete_param_formula(c,D,H,m1,m2,L=4):
    # Exact frozen cidm_architectures.py::CIDM_BER count.
    space=9*c*D + 7*D + 10*D*D
    timec=9*c*L*D + 7*D + 10*D*D
    phase=11*D*D + 3*c*D + 10*D
    fuse=4*D*D + 2*D

    skew=D*D
    flux=3*D*H + H + D
    conv_dd=10*D*H + 9*H*H + 6*H + D
    hidden=conv_dd + 1
    local=conv_dd
    core=skew+flux+hidden+local+1

    decoder=9*D*H + 9*H*H + 6*H + H*c + c
    spectral_decoder=D*D + D + D*c + c + 1

    soc=4*D*D*m1*m2 + (D*D+3*D) + (2*D*H+H+D) + 1
    M=max(8,H//2)
    ber=(27*D*H + 9*H*H + 15*H + D) + (18*D*M + 11*M + 1) + 1
    return int(
        space+timec+phase+fuse+core+decoder+spectral_decoder+soc+ber
    )

def search_cidm_complete(task):
    dim,c,S=task_spec(task)
    m1s=[1] if dim==1 else [2,4,6,8]
    max_mode=max(4,min(14,min(S)//4))
    m2s=sorted(set([4,6,8,10,12,14]))
    m2s=[m for m in m2s if m<=max(4,max_mode)]

    cands=[]
    for D in range(8,65,2):
        for H in range(16,129,4):
            for m1 in m1s:
                for m2 in m2s:
                    p=complete_param_formula(c,D,H,m1,m2)
                    cands.append((
                        abs(p-TARGET_PARAMS),p,{
                            "latent_dim":D,
                            "hidden":H,
                            "soc_modes1":m1,
                            "soc_modes2":m2,
                        }
                    ))

    err,p,cfg=min(cands,key=lambda x:x[0])
    m=build_by_config(task,"CIDM-Complete",cfg)
    actual=count_params(m)
    del m
    if actual!=p:
        raise RuntimeError(
            f"Complete formula mismatch {task}: {p} vs {actual}"
        )
    return err,p,cfg

def candidate_modes(dim,S):
    lim=max(2,min(S)//2)
    if dim==1:
        raw=[4,6,8,10,12,16,20,24,32]
    else:
        raw=[4,6,8,10,12,14,16]
    return [m for m in raw if m<=lim] or [2]

def search_official_fno(task,name):
    dim,c,S=task_spec(task)
    modes_list=candidate_modes(dim,S)
    widths=range(8,129,2) if dim==1 else range(8,81,2)
    cands=[]

    # coarse bounded search: instantiate official class, count only.
    for modes in modes_list:
        for w in widths:
            cfg={"width":w,"modes":modes}
            try:
                m=build_by_config(task,name,cfg)
                p=count_params(m)
                cands.append((abs(p-TARGET_PARAMS),p,cfg))
                del m
            except Exception:
                continue
    if not cands:
        raise RuntimeError(f"No parameter candidate for {task} {name}")
    return min(cands,key=lambda x:x[0])

def search_transolver(task):
    dim,c,S=task_spec(task)
    slice_num=32 if dim==1 else 8
    cands=[]
    for layers in [4,5,6]:
        for h in range(32,193,4):
            if h%4:
                continue
            cfg={
                "n_hidden":h,
                "n_layers":layers,
                "n_head":4,
                "slice_num":slice_num,
            }
            try:
                m=build_by_config(task,"Transolver",cfg)
                p=count_params(m)
                cands.append((abs(p-TARGET_PARAMS),p,cfg))
                del m
            except Exception:
                continue
    if not cands:
        raise RuntimeError(f"No Transolver parameter candidate for {task}")
    return min(cands,key=lambda x:x[0])

SEARCHERS={
    "CIDM-Foundation-v1":search_cidm_core,
    "CIDM-Complete":search_cidm_complete,
    "PDEBench-FNO":lambda t:search_official_fno(t,"PDEBench-FNO"),
    "NeuralOperator-FNO":lambda t:search_official_fno(t,"NeuralOperator-FNO"),
    "Transolver":search_transolver,
}

MATCH={}
rows=[]
for task in TASKS:
    MATCH[task]={}
    for name in MODELS:
        err,p,cfg=SEARCHERS[name](task)
        rel=err/TARGET_PARAMS
        ok=rel<=PARAM_TOL
        MATCH[task][name]={
            "params":p,
            "rel_error":rel,
            "ok":ok,
            "config":cfg,
        }
        rows.append({
            "task":task,
            "model":name,
            "params":p,
            "rel_error":rel,
            "ok":ok,
            "config":json.dumps(cfg),
        })
        print(
            task,name,p,
            f"{100*rel:.2f}%",
            "PASS" if ok else "FAIL",
            cfg
        )
        gc.collect()

match_df=pd.DataFrame(rows)
match_df.to_csv(OUT/"A2_parameter_matching.csv",index=False)
display(match_df)

if not match_df["ok"].all():
    raise RuntimeError(
        "A2 parameter matching failed. Formal benchmark blocked."
    )

save_json(MATCH,OUT/"A2_parameter_matching.json")
REPORT["parameter_matching"]=rows


## 7. 5/5 real-data preflight

In [ ]:

# ============================================================
# 6. Pre-flight
# ============================================================
PREFLIGHT=[]
for task in TASKS:
    dim,c,S=task_spec(task)
    batch=BATCH_1D if dim==1 else BATCH_2D
    tr,va,te=loaders_for(task,batch,DEV_SEED)
    h,y=next(iter(tr)); h,y=h.to(DEVICE),y.to(DEVICE)

    for name in MODELS:
        seed_all(DEV_SEED)
        cfg=MATCH[task][name]["config"]
        m=build_by_config(task,name,cfg).to(DEVICE).float()
        opt=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=WEIGHT_DECAY)
        ok=False; err=""
        try:
            m.train(); opt.zero_grad(set_to_none=True)
            p=m.rollout(h,HORIZON)
            if p.shape!=y.shape:
                raise RuntimeError(f"shape {p.shape} != {y.shape}")
            if not torch.isfinite(p).all():
                raise FloatingPointError("non-finite forward")
            loss=F.mse_loss(p,y)
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss")
            loss.backward()
            bad=[
                n for n,q in m.named_parameters()
                if q.grad is not None and not torch.isfinite(q.grad).all()
            ]
            if bad:
                raise FloatingPointError(f"non-finite gradients: {bad[:8]}")
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
            opt.step()
            if not all(torch.isfinite(q).all() for q in m.parameters()):
                raise FloatingPointError("non-finite params")

            sd=portable_state_dict(m)
            temp=CKPT/f"preflight__{task}__{name}.pt"
            torch.save(sd,temp)

            fresh=build_by_config(task,name,cfg).to(DEVICE).float()
            load_portable_state_dict(
                fresh,torch.load(temp,map_location="cpu",weights_only=False)
            )
            fresh.eval()
            with torch.no_grad():
                pp=fresh.rollout(h,HORIZON)
            if not torch.isfinite(pp).all():
                raise FloatingPointError("non-finite after reload")
            ok=True
        except Exception as e:
            err=repr(e)

        row={"task":task,"model":name,"pass":ok,"error":err}
        PREFLIGHT.append(row); print(row)
        del m
        if "fresh" in locals(): del fresh
        torch.cuda.empty_cache(); gc.collect()

pre_df=pd.DataFrame(PREFLIGHT)
pre_df.to_csv(OUT/"preflight.csv",index=False)
display(pre_df)
if not pre_df["pass"].all():
    raise RuntimeError("Pre-flight failed. Formal training is blocked.")
REPORT["preflight"]=PREFLIGHT


## 8. Publication metrics

In [ ]:

# ============================================================
# 7. Publication metrics
# ============================================================
def _spectral_masks(shape,device):
    if len(shape)==1:
        f=torch.fft.rfftfreq(shape[0],device=device)
        q=f/f.max().clamp_min(1e-12)
    else:
        fy=torch.fft.fftfreq(shape[-2],device=device)[:,None]
        fx=torch.fft.rfftfreq(shape[-1],device=device)[None,:]
        r=torch.sqrt(fx*fx+fy*fy)
        q=r/r.max().clamp_min(1e-12)
    return {
        "low":q<=1/3,
        "mid":(q>1/3)&(q<=2/3),
        "high":q>2/3,
    }

def _publication_spectral_metrics(pred,target):
    # pred/target [B,K,C,*S], original physical units.
    dim=pred.ndim-3
    if dim==1:
        P=torch.fft.rfft(pred.float(),dim=-1,norm="ortho").abs()
        T=torch.fft.rfft(target.float(),dim=-1,norm="ortho").abs()
        masks=_spectral_masks((pred.shape[-1],),pred.device)
    else:
        P=torch.fft.rfft2(pred.float(),dim=(-2,-1),norm="ortho").abs()
        T=torch.fft.rfft2(target.float(),dim=(-2,-1),norm="ortho").abs()
        masks=_spectral_masks((pred.shape[-2],pred.shape[-1]),pred.device)

    P2=P.square()
    T2=T.square()
    reduce_freq=(-1,) if dim==1 else (-2,-1)
    totalP=P2.sum(dim=reduce_freq).clamp_min(1e-12)
    totalT=T2.sum(dim=reduce_freq).clamp_min(1e-12)

    out={}
    logP=torch.log1p(P)
    logT=torch.log1p(T)

    for band,m in masks.items():
        # Band energy fraction: stable even if true band energy is almost zero.
        bp=P2[...,m].sum(dim=-1)
        bt=T2[...,m].sum(dim=-1)
        fp=bp/totalP
        ft=bt/totalT
        out[f"band_fraction_abs_error_{band}"]=float((fp-ft).abs().mean())

        # Log-spectrum RMSE in that band.
        d=(logP[...,m]-logT[...,m]).square().mean().sqrt()
        out[f"logspec_rmse_{band}"]=float(d)

        # Store mean true fraction for interpretability.
        out[f"true_band_fraction_{band}"]=float(ft.mean())

    return out

@torch.no_grad()
def evaluate_model(model,loader,task,max_batches=None,save_examples=False,example_tag=None):
    model.eval()
    norm=NORMALIZERS[task]
    mse=[]; mae=[]; rel=[]; leads=[]; specs=[]
    n_samples=0
    first_example=None

    torch.cuda.synchronize()
    t0=time.perf_counter()

    for bi,(h,y_n) in enumerate(loader):
        if max_batches is not None and bi>=max_batches:
            break
        h=h.to(DEVICE,non_blocking=True)
        y_n=y_n.to(DEVICE,non_blocking=True)

        p_n=model.rollout(h,HORIZON)
        p=norm.denorm_target(p_n)
        y=norm.denorm_target(y_n)
        e=p-y

        mse.append(float((e**2).mean()))
        mae.append(float(e.abs().mean()))
        rel.append(float(
            torch.linalg.vector_norm(e.reshape(e.shape[0],-1),dim=1).div(
                torch.linalg.vector_norm(y.reshape(y.shape[0],-1),dim=1).clamp_min(1e-8)
            ).mean()
        ))
        leads.append(torch.sqrt((e**2).flatten(2).mean(-1)).mean(0).cpu())
        specs.append(_publication_spectral_metrics(p,y))
        n_samples += h.shape[0]

        if first_example is None:
            # save one deterministic example in original units
            hh=norm.denorm_target(h[:, -1:, ...]).squeeze(1)
            first_example={
                "last_input":hh[0].detach().cpu(),
                "target":y[0].detach().cpu(),
                "pred":p[0].detach().cpu(),
            }

    torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0

    le=torch.stack(leads).mean(0)
    out={
        "rmse":math.sqrt(np.mean(mse)),
        "mae":float(np.mean(mae)),
        "relative_l2":float(np.mean(rel)),
        "lead_rmse":[float(x) for x in le],
        "rollout_growth":float(le[-1]/le[0].clamp_min(1e-8)),
        "eval_throughput_windows_per_s":float(n_samples/max(elapsed,1e-9)),
    }

    for k in specs[0].keys():
        out[k]=float(np.mean([x[k] for x in specs]))

    if save_examples and first_example is not None and example_tag:
        torch.save(first_example,PRED/f"{example_tag}.pt")

    return out

@torch.no_grad()
def measure_latency(model,loader,repeats=50,warmup=15):
    model.eval()
    h,_=next(iter(loader))
    h=h[:1].to(DEVICE)

    for _ in range(warmup):
        _=model.rollout(h,HORIZON)
    torch.cuda.synchronize()

    ts=[]
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0=time.perf_counter()
        _=model.rollout(h,HORIZON)
        torch.cuda.synchronize()
        ts.append((time.perf_counter()-t0)*1000)

    return {
        "latency_ms_median":float(np.median(ts)),
        "latency_ms_mean":float(np.mean(ts)),
        "latency_ms_p95":float(np.percentile(ts,95)),
    }

@torch.no_grad()
def architecture_specific_diagnostics(model,loader,name,max_batches=4):
    if name not in {"CIDM-Foundation-v1","CIDM-Complete"}:
        return {}

    rows=[]
    for bi,(h,y) in enumerate(loader):
        if bi>=max_batches: break
        h=h.to(DEVICE)

        if name=="CIDM-Foundation-v1":
            out=model.model(h,horizon=HORIZON)
            d=out["diagnostics"]
            rows.append({
                "latent_energy_drift":float(d["latent_energy_drift"].detach()),
                "local_strength":float(d["local_strength"].detach()),
                "step_size":float(d["step_size"].detach()),
            })
        else:
            rows.append(model.architecture_diagnostics(h,HORIZON))

    keys=sorted({k for r in rows for k in r})
    return {k:float(np.mean([r[k] for r in rows if k in r])) for k in keys}


## 9. Frozen global LR

In [ ]:

LR_SELECTED={
    "CIDM-Foundation-v1":1e-4,
    "CIDM-Complete":1e-3,
    "PDEBench-FNO":3e-3,
    "NeuralOperator-FNO":3e-3,
    "Transolver":3e-4,
}
assert set(LR_SELECTED)==set(MODELS)
save_json(LR_SELECTED,OUT/"A2_R1_selected_lr.json")



## 10. 15-run formal repair benchmark

只有真正的 1D Diff-Sorp 重跑：

\[
5\times3=15
\]

每 run 完成后立即上传 HF Dataset。


In [ ]:

# ============================================================
# 10. Deterministic 15-run repair training
# ============================================================
def normalized_split_tensors(task,split):
    c=CACHES[task]; norm=NORMALIZERS[task]
    h=c["splits"][split]["history"].float()
    y=c["splits"][split]["target"].float()
    mh=norm.mean.view(1,1,-1,*([1]*(h.ndim-3)))
    sh=norm.std.view(1,1,-1,*([1]*(h.ndim-3)))
    return ((h-mh)/sh).contiguous(),((y-mh)/sh).contiguous()

TRAIN_TENSORS={TASK:normalized_split_tensors(TASK,"train")}

def deterministic_batch_plan(n,batch,seed,steps):
    rng=np.random.default_rng(seed)
    plan=[]
    while len(plan)<steps:
        perm=rng.permutation(n)
        usable=(len(perm)//batch)*batch
        for j in range(0,usable,batch):
            plan.append(perm[j:j+batch].astype(np.int64))
            if len(plan)>=steps:
                break
    return plan

def train_to_convergence(model,task,va,lr,seed,tag,latest_path):
    model=model.to(DEVICE).float()
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,T_max=MAX_UPDATES,eta_min=lr*0.1
    )
    train_h,train_y=TRAIN_TENSORS[task]
    plan=deterministic_batch_plan(len(train_h),BATCH_1D,seed,MAX_UPDATES)

    start_step=0
    raw_best=float("inf"); raw_best_step=0; best_sd=None
    convergence_ref=float("inf"); bad_evals=0; history=[]; elapsed_before=0.0

    if latest_path.exists():
        ck=torch.load(latest_path,map_location="cpu",weights_only=False)
        load_portable_state_dict(model,ck["model"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        start_step=int(ck["step"])
        raw_best=float(ck["raw_best"])
        raw_best_step=int(ck["raw_best_step"])
        best_sd=ck.get("best_model")
        convergence_ref=float(ck.get("convergence_ref",raw_best))
        bad_evals=int(ck.get("bad_evals",0))
        history=list(ck.get("history",[]))
        elapsed_before=float(ck.get("elapsed_train_seconds",0.0))
        print(f"[A2-R1 resume] {tag} step={start_step}")

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0=time.perf_counter()
    stop_reason="MAX_UPDATES"

    for s in range(start_step+1,MAX_UPDATES+1):
        idx=torch.as_tensor(plan[s-1],dtype=torch.long)
        h=train_h.index_select(0,idx).to(DEVICE,non_blocking=True)
        y=train_y.index_select(0,idx).to(DEVICE,non_blocking=True)

        model.train(); opt.zero_grad(set_to_none=True)
        p=model.rollout(h,HORIZON)
        loss=F.mse_loss(p,y)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"{tag}: nonfinite loss step {s}")
        loss.backward()
        bad=[
            n for n,q in model.named_parameters()
            if q.grad is not None and not torch.isfinite(q.grad).all()
        ]
        if bad:
            raise FloatingPointError(f"{tag}: nonfinite gradients {bad[:8]}")
        gn=float(torch.nn.utils.clip_grad_norm_(model.parameters(),1.0))
        opt.step(); sched.step()

        should_stop=False
        if s==1 or s%EVAL_EVERY==0 or s==MAX_UPDATES:
            vr=evaluate_model(model,va,task)["rmse"]
            if vr<raw_best:
                raw_best=vr; raw_best_step=s; best_sd=portable_state_dict(model)
            if not math.isfinite(convergence_ref):
                convergence_ref=vr; bad_evals=0
            else:
                if vr<convergence_ref*(1-REL_MIN_DELTA):
                    convergence_ref=vr; bad_evals=0
                elif s>=MIN_UPDATES:
                    bad_evals+=1
            history.append({
                "step":s,
                "train_normalized_mse":float(loss.detach()),
                "val_rmse_original":float(vr),
                "raw_best_val_rmse":float(raw_best),
                "raw_best_step":int(raw_best_step),
                "convergence_ref":float(convergence_ref),
                "bad_evals":int(bad_evals),
                "lr":float(opt.param_groups[0]["lr"]),
                "grad_norm_preclip":gn,
            })
            print(
                f"[{tag}] {s}/{MAX_UPDATES} val={vr:.6g} "
                f"best={raw_best:.6g}@{raw_best_step} "
                f"bad={bad_evals}/{PATIENCE_EVALS}"
            )
            if s>=MIN_UPDATES and bad_evals>=PATIENCE_EVALS:
                stop_reason="CONVERGED_PATIENCE"
                should_stop=True

        if s%100==0 or s==MAX_UPDATES or should_stop:
            torch.cuda.synchronize()
            elapsed=elapsed_before+(time.perf_counter()-t0)
            torch.save({
                "step":s,
                "model":portable_state_dict(model),
                "best_model":best_sd,
                "raw_best":raw_best,
                "raw_best_step":raw_best_step,
                "convergence_ref":convergence_ref,
                "bad_evals":bad_evals,
                "optimizer":opt.state_dict(),
                "scheduler":sched.state_dict(),
                "history":history,
                "elapsed_train_seconds":elapsed,
            },latest_path)
            if HF_ENABLED and (
                s%HF_CHECKPOINT_EVERY==0 or s==MAX_UPDATES or should_stop
            ):
                hf_sync_checkpoint(latest_path,tag,s)

        if should_stop:
            break

    torch.cuda.synchronize()
    elapsed=elapsed_before+(time.perf_counter()-t0)
    peak=int(torch.cuda.max_memory_allocated())
    actual=history[-1]["step"] if history else start_step
    if best_sd is None:
        raw_best=evaluate_model(model,va,task)["rmse"]
        raw_best_step=actual
        best_sd=portable_state_dict(model)
    load_portable_state_dict(model,best_sd)

    return model,{
        "best_val_rmse":float(raw_best),
        "best_val_step":int(raw_best_step),
        "actual_updates":int(actual),
        "stop_reason":stop_reason,
        "train_gpu_seconds":float(elapsed),
        "peak_vram_bytes":peak,
        "history":history,
    }

FORMAL=[]
for seed in SEEDS:
    _,va,te=loaders_for(TASK,BATCH_1D,seed)
    for name in MODELS:
        run_id=f"A2R1__{name}__{TASK}__s{seed}"
        result_path=OUT/f"{run_id}_result.json"
        history_path=OUT/f"{run_id}_history.csv"
        latest_path=CKPT/f"{run_id}_latest.pt"

        if result_path.exists():
            try:
                row=json.loads(result_path.read_text(encoding="utf-8"))
            except Exception:
                row={}
            if row.get("status")=="COMPLETED":
                FORMAL.append(row)
                print("[A2-R1 skip]",run_id)
                continue

        seed_all(seed)
        cfg=MATCH[TASK][name]["config"]
        m=build_by_config(TASK,name,cfg)
        lr=LR_SELECTED[name]
        print("\n"+"="*100)
        print(run_id,count_params(m),cfg)

        try:
            m,ti=train_to_convergence(
                m,TASK,va,lr,seed,run_id,latest_path
            )
            pd.DataFrame(ti["history"]).to_csv(history_path,index=False)
            ev=evaluate_model(
                m,te,TASK,
                save_examples=(seed==SEEDS[0]),
                example_tag=f"{TASK}__{name}"
            )
            lat=measure_latency(m,te)
            diag=architecture_specific_diagnostics(m,te,name)

            row={
                "seed":seed,
                "task":TASK,
                "family":TASKS[TASK]["family"],
                "model":name,
                "status":"COMPLETED",
                "params":count_params(m),
                "lr":lr,
                "data_contract_version":"A2-R1-FIX1-TRUE-1D",
                "best_val_rmse":ti["best_val_rmse"],
                "best_val_step":ti["best_val_step"],
                "actual_updates":ti["actual_updates"],
                "stop_reason":ti["stop_reason"],
                "train_gpu_seconds":ti["train_gpu_seconds"],
                "peak_vram_bytes":ti["peak_vram_bytes"],
                **ev,**lat,**diag,
            }
            save_json(row,result_path)
        except Exception as e:
            row={
                "seed":seed,"task":TASK,
                "family":TASKS[TASK]["family"],
                "model":name,"status":"FAILED",
                "params":count_params(m),"lr":lr,
                "error":repr(e),
            }
            save_json(row,result_path)

        FORMAL.append(row)
        hf_sync_finished(run_id,result_path,history_path,latest_path)
        if row.get("status")=="COMPLETED":
            latest_path.unlink(missing_ok=True)
        del m
        torch.cuda.empty_cache(); gc.collect()

# rebuild from disk
fix_rows=[]
for p in sorted(OUT.glob("A2R1__*_result.json")):
    try:
        fix_rows.append(json.loads(p.read_text(encoding="utf-8")))
    except Exception:
        pass
fix_df=pd.DataFrame(fix_rows)
fix_df.to_csv(OUT/"A2_R1_FIX_run_results.csv",index=False)
display(fix_df)

completed=int((fix_df.status=="COMPLETED").sum()) if len(fix_df) else 0
if completed!=15:
    raise RuntimeError(
        f"A2-R1 repair incomplete {completed}/15. Restart Colab; HF resumes."
    )
REPORT["repair_runs"]=fix_df.to_dict("records")



## 11. Merge 105 valid A2 runs + 15 repaired Diff-Sorp runs

原错误任务不会参与 corrected aggregate。


In [ ]:

# ============================================================
# 11. Corrected full A2 merge
# ============================================================
base_df=pd.DataFrame(BASE_A2_ROWS)
if len(base_df)!=120:
    raise RuntimeError(
        f"Expected 120 completed Base A2 result JSONs from HF, got {len(base_df)}."
    )

base_valid=base_df[
    (base_df.status=="COMPLETED")
    &(base_df.task!="diff_sorp_1d")
].copy()

if len(base_valid)!=105:
    raise RuntimeError(
        f"Expected 105 non-DiffSorp valid Base A2 runs, got {len(base_valid)}."
    )

fix_valid=fix_df[fix_df.status=="COMPLETED"].copy()
assert len(fix_valid)==15

corrected=pd.concat([base_valid,fix_valid],ignore_index=True)
assert len(corrected)==120

# Unique task/model/seed contract.
dups=corrected.duplicated(["task","model","seed"]).sum()
assert dups==0,dups

corrected.to_csv(
    OUT/"A2_R1_CORRECTED_run_results.csv",
    index=False
)

summary=corrected.groupby(["task","family","model"]).agg(
    rmse_mean=("rmse","mean"),
    rmse_std=("rmse","std"),
    mae_mean=("mae","mean"),
    relative_l2_mean=("relative_l2","mean"),
    rollout_growth_mean=("rollout_growth","mean"),
    logspec_low_mean=("logspec_rmse_low","mean"),
    logspec_mid_mean=("logspec_rmse_mid","mean"),
    logspec_high_mean=("logspec_rmse_high","mean"),
    updates_mean=("actual_updates","mean"),
    train_seconds_mean=("train_gpu_seconds","mean"),
    latency_ms_mean=("latency_ms_median","mean"),
    peak_vram_mean=("peak_vram_bytes","mean"),
).reset_index()

mins=summary.groupby("task")["rmse_mean"].min().to_dict()
summary["task_normalized_rmse_ratio"]=summary.apply(
    lambda r:r.rmse_mean/mins[r.task],
    axis=1
)

overall=summary.groupby("model").agg(
    mean_task_normalized_rmse=("task_normalized_rmse_ratio","mean"),
    median_task_normalized_rmse=("task_normalized_rmse_ratio","median"),
    mean_rollout_growth=("rollout_growth_mean","mean"),
    mean_logspec_high=("logspec_high_mean","mean"),
    mean_latency_ms=("latency_ms_mean","mean"),
    mean_train_seconds=("train_seconds_mean","mean"),
    mean_peak_vram=("peak_vram_mean","mean"),
).reset_index().sort_values("mean_task_normalized_rmse")

regime=summary[[
    "task","family","model","rmse_mean","rmse_std",
    "task_normalized_rmse_ratio"
]].copy()
regime["rank_within_task"]=regime.groupby("task")[
    "rmse_mean"
].rank(method="min")

summary.to_csv(OUT/"A2_R1_CORRECTED_summary.csv",index=False)
overall.to_csv(OUT/"A2_R1_CORRECTED_overall.csv",index=False)
regime.to_csv(OUT/"A2_R1_CORRECTED_regime_map.csv",index=False)

# Core vs Complete descriptive.
cc=[]
for task in corrected.task.unique():
    a=corrected[
        (corrected.task==task)
        &(corrected.model=="CIDM-Foundation-v1")
    ][["seed","rmse"]].rename(columns={"rmse":"core"})
    b=corrected[
        (corrected.task==task)
        &(corrected.model=="CIDM-Complete")
    ][["seed","rmse"]].rename(columns={"rmse":"complete"})
    m=a.merge(b,on="seed")
    for _,r in m.iterrows():
        cc.append({
            "task":task,
            "seed":int(r.seed),
            "core_rmse":float(r.core),
            "complete_rmse":float(r.complete),
            "complete_improvement_pct":
                100*(r.core-r.complete)/r.core,
        })
pd.DataFrame(cc).to_csv(
    OUT/"A2_R1_CORRECTED_Core_vs_Complete.csv",
    index=False
)

print("CORRECTED A2 overall:")
display(overall)
print("CORRECTED A2 task ranking:")
display(regime.sort_values(["task","rank_within_task"]))
REPORT["corrected_overall"]=overall.to_dict("records")


## 12. Corrected A2 publication figures

In [ ]:

# ============================================================
# 12. Corrected A2 figures
# ============================================================
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor":"white",
    "axes.facecolor":"white",
    "savefig.facecolor":"white",
    "font.family":"serif",
    "font.serif":["Times New Roman","Nimbus Roman","DejaVu Serif"],
    "mathtext.fontset":"stix",
    "font.size":9,
    "axes.linewidth":0.8,
    "xtick.direction":"in",
    "ytick.direction":"in",
    "legend.frameon":False,
    "pdf.fonttype":42,
    "ps.fonttype":42,
})

MODEL_ORDER=[
    "CIDM-Foundation-v1",
    "CIDM-Complete",
    "PDEBench-FNO",
    "NeuralOperator-FNO",
    "Transolver",
]
MODEL_LABEL={
    "CIDM-Foundation-v1":"CIDM-Core",
    "CIDM-Complete":"CIDM-Complete",
    "PDEBench-FNO":"PDEBench FNO",
    "NeuralOperator-FNO":"NeuralOperator FNO",
    "Transolver":"Transolver",
}
MODEL_COLOR={
    "CIDM-Foundation-v1":"#1f4e79",
    "CIDM-Complete":"#b2182b",
    "PDEBench-FNO":"#4d4d4d",
    "NeuralOperator-FNO":"#2b8cbe",
    "Transolver":"#6a51a3",
}
TASK_ORDER=[
    "advection_1d","burgers_1d","diff_sorp_1d","reacdiff_1d",
    "cfd_1d","reacdiff_2d","swe_2d","cfd_2d"
]
TASK_LABEL={
    "advection_1d":"Advection",
    "burgers_1d":"Burgers",
    "diff_sorp_1d":"Diff.-Sorption",
    "reacdiff_1d":"1D Reac.-Diff.",
    "cfd_1d":"1D CFD",
    "reacdiff_2d":"2D Reac.-Diff.",
    "swe_2d":"Shallow Water",
    "cfd_2d":"2D CFD",
}

def savefig_all(fig,stem):
    for ext in ["pdf","svg","png"]:
        kw={"bbox_inches":"tight"}
        if ext=="png": kw["dpi"]=600
        fig.savefig(FIG/f"{stem}.{ext}",**kw)
    plt.close(fig)

# Heatmap
mat=regime.pivot(
    index="model",columns="task",
    values="task_normalized_rmse_ratio"
).loc[MODEL_ORDER,TASK_ORDER]

fig,ax=plt.subplots(figsize=(8.2,2.7))
im=ax.imshow(
    mat.values,aspect="auto",cmap="YlOrRd",
    vmin=1.0,
    vmax=max(2.5,float(np.nanpercentile(mat.values,90)))
)
ax.set_xticks(np.arange(len(TASK_ORDER)))
ax.set_xticklabels(
    [TASK_LABEL[t] for t in TASK_ORDER],
    rotation=35,ha="right"
)
ax.set_yticks(np.arange(len(MODEL_ORDER)))
ax.set_yticklabels([MODEL_LABEL[m] for m in MODEL_ORDER])
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v=mat.iloc[i,j]
        ax.text(
            j,i,f"{v:.2f}",
            ha="center",va="center",fontsize=7,
            color="white" if v>1.8 else "black"
        )
cb=fig.colorbar(im,ax=ax,pad=0.015)
cb.set_label("Task-normalized RMSE")
ax.set_title("Corrected broad dynamics regime map")
fig.tight_layout()
mat.reset_index().to_csv(
    FIGDATA/"Fig_A2_R1_01_corrected_regime_heatmap.csv",
    index=False
)
savefig_all(fig,"Fig_A2_R1_01_corrected_regime_heatmap")

# 8 task panels
fig,axes=plt.subplots(2,4,figsize=(10.5,5.0))
axes=axes.ravel()
for ax,task in zip(axes,TASK_ORDER):
    q=summary[summary.task==task].set_index("model").loc[MODEL_ORDER]
    x=np.arange(len(MODEL_ORDER))
    ax.bar(
        x,q.rmse_mean.values,
        color=[MODEL_COLOR[m] for m in MODEL_ORDER],
        edgecolor="black",linewidth=0.35,width=0.72
    )
    ax.errorbar(
        x,q.rmse_mean.values,yerr=q.rmse_std.values,
        fmt="none",ecolor="black",capsize=2,elinewidth=0.7
    )
    for j,m in enumerate(MODEL_ORDER):
        vals=corrected[
            (corrected.task==task)&(corrected.model==m)
        ].rmse.values
        ax.scatter(
            np.full(len(vals),j)+np.linspace(-.08,.08,len(vals)),
            vals,s=9,facecolor="white",edgecolor="black",
            linewidth=.45,zorder=5
        )
    ax.set_yscale("log")
    ax.set_title(TASK_LABEL[task])
    ax.set_xticks(x)
    ax.set_xticklabels(
        [MODEL_LABEL[m] for m in MODEL_ORDER],
        rotation=55,ha="right"
    )
    ax.set_ylabel("RMSE")
fig.tight_layout()
savefig_all(fig,"Fig_A2_R1_02_corrected_task_panels")

# Corrected overall Pareto
fig,ax=plt.subplots(figsize=(3.9,3.0))
for _,r in overall.iterrows():
    m=r.model
    ax.scatter(
        r.mean_latency_ms,r.mean_task_normalized_rmse,
        s=46,color=MODEL_COLOR[m],
        edgecolor="black",linewidth=.45
    )
    ax.annotate(
        MODEL_LABEL[m],
        (r.mean_latency_ms,r.mean_task_normalized_rmse),
        xytext=(4,3),textcoords="offset points",fontsize=7
    )
ax.set_xlabel("Median 6-step latency (ms)")
ax.set_ylabel("Mean task-normalized RMSE")
fig.tight_layout()
savefig_all(fig,"Fig_A2_R1_03_corrected_accuracy_latency")

print("Corrected figures:",FIG)


## 13. Repair gate

In [ ]:

# ============================================================
# 13. Final repair gate
# ============================================================
dim,ch,sp=task_spec("diff_sorp_1d")
checks={
    "official_diff_sorp_true_1d_contract":
        dim==1 and ch==1 and list(sp)==[256],
    "parameter_match_5of5":
        len(match_df)==5 and bool(match_df["ok"].all()),
    "preflight_5of5":
        len(pre_df)==5 and bool(pre_df["pass"].all()),
    "repair_15of15":
        len(fix_df)==15 and bool((fix_df.status=="COMPLETED").all()),
    "base_valid_105of105":
        len(base_valid)==105,
    "corrected_merged_120of120":
        len(corrected)==120
        and corrected.duplicated(["task","model","seed"]).sum()==0,
    "corrected_diff_sorp_15of15":
        len(corrected[corrected.task=="diff_sorp_1d"])==15,
    "old_invalid_diff_sorp_excluded":
        not (
            "data_contract_version" not in corrected.columns
            or corrected[
                corrected.task=="diff_sorp_1d"
            ]["data_contract_version"].fillna("").ne(
                "A2-R1-FIX1-TRUE-1D"
            ).any()
        ),
    "corrected_figures_written":
        all(
            (FIG/f"{stem}.{ext}").exists()
            for stem in [
                "Fig_A2_R1_01_corrected_regime_heatmap",
                "Fig_A2_R1_02_corrected_task_panels",
                "Fig_A2_R1_03_corrected_accuracy_latency",
            ]
            for ext in ["pdf","svg","png"]
        ),
}

verdict=(
    "PASS_A2_R1_DIFFSORP_1D_REPAIR_AND_CORRECTED_MERGE"
    if all(checks.values())
    else "A2_R1_REPAIR_INCOMPLETE"
)

REPORT["gate"]=checks
REPORT["verdict"]=verdict
REPORT["finished_at"]=time.strftime("%Y-%m-%d %H:%M:%S")
save_json(REPORT,OUT/"CIDM_Foundation_A2_R1_REPORT.json")

lines=[
    "CIDM Foundation A2-R1 FIX1 — Diffusion-Sorption 1D Contract Repair",
    "="*88,
    f"VERDICT: {verdict}",
    "",
]
for k,v in checks.items():
    lines.append(f"[{'PASS' if v else 'FAIL'}] {k}")
lines += [
    "",
    "Only the invalid Diffusion-Sorption task was rerun.",
    "The other 105 valid A2 runs were preserved unchanged.",
    "",
    "Next if PASS: A3 CIDM-Complete Component Attribution.",
]
gate=OUT/"A2_R1_gate_report.txt"
gate.write_text("\n".join(lines),encoding="utf-8")
print(gate.read_text(encoding="utf-8"))


## 14. Package + final HF upload

In [ ]:

# ============================================================
# 14. Package + final HF snapshot
# ============================================================
zip_path=OUT/"CIDM_Foundation_A2_R1_CORRECTED_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if not p.is_file() or p==zip_path:
            continue
        if "sources" in p.parts or "checkpoints" in p.parts:
            continue
        zf.write(p,p.relative_to(OUT))

sha=sha256_file(zip_path)
sha_path=OUT/"CIDM_Foundation_A2_R1_CORRECTED_RESULTS.zip.sha256"
sha_path.write_text(sha+"\n",encoding="utf-8")
print("ZIP:",zip_path)
print("SHA256:",sha)

if HF_ENABLED:
    try:
        write_fix_manifest(None,None,"FINAL_PACKAGE")
        HF_API.upload_folder(
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            folder_path=str(OUT),
            path_in_repo=HF_FIX_PREFIX,
            ignore_patterns=[
                "sources/*",
                "checkpoints/*",
                "base_A2_results/*",
            ],
            commit_message="[A2-R1] final corrected broad-dynamics package",
        )
        print(
            f"[HF FINAL] https://huggingface.co/datasets/"
            f"{HF_DATASET_REPO_ID}/tree/main/{HF_FIX_PREFIX}"
        )
    except Exception as e:
        print("[HF final warning]",repr(e))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Auto-download unavailable outside Colab:",repr(e))
    print("Result:",zip_path.resolve())
